# Monitor: coverage-first visualizations

Run 01-04 with curated writes enabled first. This notebook is read-only and
never starts collectors. Charts count observations unless labelled otherwise.
Retained job status is not proof of current polling health. A permission view
uses one explicit mode and cycle; a lineage view uses latest resource observations
or one exact historical source. Failures never fall back silently to old edges.
Principal IDs, names, UPNs, raw definitions and failure messages are not chart labels.


In [ ]:
# Attach the SAME default Lakehouse and use the same prefix as 01-04.
TABLE_PREFIX = "monitor_"
PERMISSION_MODE = "Scoped"  # Explicitly choose "Scoped" or "Tenant".
PERMISSION_CYCLE_ID = ""  # Empty selects latest attempt in the chosen mode.
STALE_AFTER_HOURS = 24
LINEAGE_SOURCE_PATH = ""  # Optional exact historical batch/resource; never a union.
MAX_GRAPH_EDGES = 40

In [ ]:
"""Embedded Monitor implementation; generated from tested repository sources."""

import hashlib
import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from urllib.parse import urlsplit
from uuid import UUID, uuid4

ITEM_TYPES = {"reports": "Report", "datasets": "SemanticModel", "dataflows": "Dataflow"}


def _text(value, field, required=False):
    if value is None and not required:
        return None
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"Invalid lineage field: {field}")
    return value.strip()


def _identifier(value, field, required=False):
    text = _text(value, field, required)
    return text.lower() if text else None


def _records(parent, field, required=False):
    value = parent.get(field)
    if value is None and not required:
        return []
    if not isinstance(value, list) or any(not isinstance(row, dict) for row in value):
        raise ValueError(f"Invalid lineage collection: {field}")
    return value


def scan_snapshot(
    scan, scan_id, requested_workspace_ids, observed_at_utc, full_scan=False
):
    """Build a batch observation, not an exhaustive or historical tenant graph.

    Unknown fields are ignored. Malformed recognized fields raise ``ValueError``
    rather than becoming successful empty observations. Missing optional
    metadata and references outside this batch remain unresolved.
    """
    if not isinstance(scan_id, str) or not re.fullmatch(
        r"[A-Za-z0-9][A-Za-z0-9_-]{0,127}", scan_id
    ):
        raise ValueError("Invalid lineage scan ID")
    if not isinstance(scan, dict) or any(key in scan for key in ("ERROR", "error")):
        raise ValueError("Invalid lineage scan response")
    requested = sorted(
        {
            _identifier(value, "requestedWorkspaceId", required=True)
            for value in requested_workspace_ids
        }
    )
    workspaces = _records(scan, "workspaces", required=True)
    returned = set()
    items = {}
    raw_items = {}
    for workspace in workspaces:
        ws_id = _identifier(workspace.get("id"), "workspace.id", required=True)
        if ws_id in returned or ws_id not in requested:
            raise ValueError("Duplicate or unexpected lineage workspace")
        if any(key in workspace for key in ("ERROR", "error", "scanError")):
            raise ValueError("Failed lineage workspace")
        returned.add(ws_id)
        for key, kind in ITEM_TYPES.items():
            for raw in _records(workspace, key):
                item_id = _identifier(
                    raw.get("objectId") if key == "dataflows" else raw.get("id"),
                    f"{key}.id",
                    required=True,
                )
                if item_id in items:
                    raise ValueError("Duplicate lineage item")
                items[item_id] = {
                    "id": item_id,
                    "name": _text(raw.get("name"), f"{key}.name"),
                    "type": kind,
                    "workspaceId": ws_id,
                }
                raw_items[item_id] = raw

    datasources = {}
    for raw in _records(scan, "datasourceInstances"):
        source_id = _identifier(raw.get("datasourceId"), "datasourceId", required=True)
        if source_id in datasources:
            raise ValueError("Duplicate lineage datasource")
        kind = _text(raw.get("datasourceType"), "datasourceType")
        details = raw.get("connectionDetails")
        if details is not None and not isinstance(details, dict):
            raise ValueError("Invalid lineage connectionDetails")
        # Do not export arbitrary connection strings, URLs or expression payloads.
        labels = [
            _text((details or {}).get(key), f"connectionDetails.{key}")
            for key in ("server", "database")
        ]
        label = " / ".join(part for part in labels if part)
        datasources[source_id] = {
            "id": source_id,
            "name": f"{kind}: {label}" if kind and label else (kind or label or None),
            "type": "DataSource",
            "workspaceId": None,
        }

    edges = {}

    def add(query_id, related_id, relation, related_type=None, related_ws=None):
        query = items[query_id]
        related = (
            datasources.get(related_id)
            if related_type == "DataSource"
            else items.get(related_id)
        )
        if related is None:
            related = {
                "id": related_id,
                "name": None,
                "type": related_type,
                "workspaceId": related_ws,
            }

        def emit(source, target, direction):
            row = {
                "QueryItemId": source["id"],
                "QueryItemName": source["name"],
                "QueryItemType": source["type"],
                "QueryWorkspaceId": source["workspaceId"],
                "Direction": direction,
                "RelatedItemId": target["id"],
                "RelatedItemName": target["name"],
                "RelatedItemType": target["type"],
                "RelatedWorkspaceId": target["workspaceId"],
                "RelationType": relation,
                "Source": "Scanner",
            }
            key = (source["id"], direction, target["id"], relation)
            existing = edges.get(key)
            if existing is not None:
                for field, value in row.items():
                    if existing[field] is None:
                        existing[field] = value
                    elif value is not None and existing[field] != value:
                        raise ValueError("Conflicting lineage reference metadata")
            else:
                edges[key] = row

        emit(query, related, "upstream")
        if related_type != "DataSource" and related_id in items:
            emit(related, query, "downstream")

    for item_id, raw in raw_items.items():
        if items[item_id]["type"] == "Report" and raw.get("datasetId") is not None:
            add(
                item_id,
                _identifier(raw["datasetId"], "datasetId", True),
                "ReportModel",
                "SemanticModel",
            )
        for field, target, relation, kind in (
            ("upstreamDatasets", "targetDatasetId", "UpstreamDataset", "SemanticModel"),
            ("upstreamDataflows", "targetDataflowId", "UpstreamDataflow", "Dataflow"),
        ):
            for ref in _records(raw, field):
                add(
                    item_id,
                    _identifier(ref.get(target), target, True),
                    relation,
                    kind,
                    _identifier(ref.get("groupId"), "groupId"),
                )
        for ref in _records(raw, "relations"):
            add(
                item_id,
                _identifier(
                    ref.get("dependentOnArtifactId"), "dependentOnArtifactId", True
                ),
                _text(ref.get("relationType"), "relationType") or "Relation",
            )
        for ref in _records(raw, "datasourceUsages"):
            add(
                item_id,
                _identifier(
                    ref.get("datasourceInstanceId"), "datasourceInstanceId", True
                ),
                "DataSource",
                "DataSource",
            )

    return {
        "schemaVersion": 1,
        "source": "Scanner",
        "scanId": scan_id,
        "observedAtUtc": observed_at_utc,
        "scanMode": "full" if full_scan else "incremental",
        "requestedWorkspaceIds": requested,
        "returnedWorkspaceIds": sorted(returned),
        "items": [items[key] for key in sorted(items)],
        "complete": True,
        "edges": [edges[key] for key in sorted(edges)],
    }


def notebook_lakehouse_relationships(notebook):
    """Return configured identifiers only; never resolve names or check access."""
    relationships, diagnostics = [], []

    def diagnostic(path, reason):
        diagnostics.append({"fieldPath": path, "reason": reason})

    def identifier(value, path, *, optional=False):
        if optional and value in (None, ""):
            return None
        try:
            if not isinstance(value, str):
                raise TypeError
            return str(UUID(value))
        except (TypeError, ValueError):
            diagnostic(path, "invalid_configured_identifier")
            return None

    path = "/metadata"
    value = notebook
    for key in ("metadata", "dependencies", "lakehouse"):
        if not isinstance(value, dict):
            diagnostic(path, "invalid_notebook_configuration")
            break
        value = value.get(key, {})
        if key != "metadata" and value is None:
            value = {}
        path = "/metadata" if key == "metadata" else path + "/" + key
    else:
        if not isinstance(value, dict):
            diagnostic(path, "invalid_notebook_configuration")
        else:
            default_id = identifier(
                value.get("default_lakehouse"),
                path + "/default_lakehouse",
                optional=True,
            )
            default_workspace = identifier(
                value.get("default_lakehouse_workspace_id"),
                path + "/default_lakehouse_workspace_id",
                optional=True,
            )
            if default_workspace and not default_id:
                diagnostic(path + "/default_lakehouse", "missing_configured_lakehouse")
            if default_id:
                relationships.append(
                    {
                        "relation": "default_lakehouse",
                        "itemId": default_id,
                        "workspaceId": default_workspace,
                    }
                )
            known = value.get("known_lakehouses", [])
            if not isinstance(known, list):
                diagnostic(path + "/known_lakehouses", "invalid_notebook_configuration")
            else:
                seen = {default_id: default_workspace} if default_id else {}
                for index, entry in enumerate(known):
                    location = f"{path}/known_lakehouses/{index}"
                    if not isinstance(entry, dict):
                        diagnostic(location, "invalid_notebook_configuration")
                        continue
                    item_id = identifier(entry.get("id"), location + "/id")
                    workspace = identifier(
                        entry.get("workspaceId"),
                        location + "/workspaceId",
                        optional=True,
                    )
                    if item_id is None:
                        continue
                    if item_id in seen:
                        if seen[item_id] and workspace and seen[item_id] != workspace:
                            diagnostic(location, "conflicting_configured_workspace")
                        elif workspace and not seen[item_id]:
                            seen[item_id] = workspace
                            for relationship in relationships:
                                if relationship["itemId"] == item_id:
                                    relationship["workspaceId"] = workspace
                        continue
                    seen[item_id] = workspace
                    relationships.append(
                        {
                            "relation": "known_lakehouse",
                            "itemId": item_id,
                            "workspaceId": workspace,
                        }
                    )
    return {
        "schemaVersion": 1,
        "evidence": "configuration_metadata",
        "runtimeVerified": False,
        "complete": not diagnostics,
        "relationships": relationships,
        "diagnostics": diagnostics,
    }


SHORTCUT_MODULE = "fabricshortcuts"
_SHORTCUT_TYPES = {
    "AdlsGen2": "adlsGen2",
    "AmazonS3": "amazonS3",
    "AzureBlobStorage": "azureBlobStorage",
    "Dataverse": "dataverse",
    "ExternalDataShare": "externalDataShare",
    "GoogleCloudStorage": "googleCloudStorage",
    "OneDriveSharePoint": "oneDriveSharePoint",
    "S3Compatible": "s3Compatible",
}
_SHORTCUT_FIELDS = {
    "shortcutId",
    "targetType",
    "targetStatus",
    "targetWorkspaceId",
    "targetItemId",
    "targetReference",
    "isShortcutTransform",
}


def shortcut_fingerprint(value):
    return hashlib.sha256(
        json.dumps(
            value, sort_keys=True, separators=(",", ":"), allow_nan=False
        ).encode()
    ).hexdigest()


def shortcut_exclusions(value):
    if not isinstance(value, str):
        raise TypeError(
            "Shortcut workspace exclusions must be a comma-separated string"
        )
    try:
        return sorted(
            {str(UUID(part.strip())) for part in value.split(",") if part.strip()}
        )
    except (ValueError, AttributeError):
        raise ValueError("Shortcut workspace exclusions require UUIDs") from None


def shortcut_inventory(values, excluded):
    if not isinstance(values, list):
        raise TypeError("Invalid shortcut inventory")
    selected = {}
    for value in values:
        if not isinstance(value, dict) or value.get("type") != "Lakehouse":
            raise ValueError("Invalid shortcut inventory type")
        parent = {
            "workspaceId": str(UUID(value["workspaceId"])),
            "itemId": str(UUID(value["id"])),
            "type": "Lakehouse",
        }
        if parent["workspaceId"] not in excluded:
            selected[parent["workspaceId"], parent["itemId"]] = parent
    return [selected[key] for key in sorted(selected)]


def validate_shortcut_records(values):
    if not isinstance(values, list):
        raise TypeError("Invalid shortcut records")
    seen = set()
    for value in values:
        if not isinstance(value, dict) or set(value) != _SHORTCUT_FIELDS:
            raise ValueError("Invalid shortcut record fields")
        identity = value["shortcutId"]
        if (
            not isinstance(identity, str)
            or re.fullmatch(r"[a-f0-9]{64}", identity) is None
            or identity in seen
            or not isinstance(value["targetType"], str)
            or re.fullmatch(r"[A-Za-z][A-Za-z0-9]{0,63}", value["targetType"]) is None
            or value["targetStatus"]
            not in {
                "configured_item",
                "external_reference",
                "unavailable",
                "unsupported",
            }
            or (
                value["isShortcutTransform"] is not None
                and type(value["isShortcutTransform"]) is not bool
            )
        ):
            raise ValueError("Invalid shortcut identity or status")
        seen.add(identity)
        configured = value["targetStatus"] == "configured_item"
        for field in ("targetWorkspaceId", "targetItemId"):
            target_id = value[field]
            if configured:
                if str(UUID(target_id)) != target_id:
                    raise ValueError("Invalid shortcut target identity")
            elif target_id is not None:
                raise ValueError("Unresolved shortcut has an item identity")
        reference = value["targetReference"]
        if reference is not None and (
            not isinstance(reference, str)
            or re.fullmatch(r"[a-f0-9]{64}", reference) is None
        ):
            raise ValueError("Invalid shortcut target fingerprint")
        if (
            configured
            and value["targetType"] != "OneLake"
            or value["targetStatus"] == "external_reference"
            and (value["targetType"] not in _SHORTCUT_TYPES or reference is None)
            or value["targetStatus"] in {"unavailable", "unsupported"}
            and reference is not None
        ):
            raise ValueError("Shortcut target type and resolution disagree")
    return values


def shortcut_records(values):
    """Drop names, paths, URLs, credentials and arbitrary response properties."""
    if not isinstance(values, list):
        raise TypeError("Invalid shortcut list")
    selected = {}
    for value in values:
        if not isinstance(value, dict):
            raise TypeError("Invalid shortcut")
        path, name, target = value.get("path"), value.get("name"), value.get("target")
        if (
            not isinstance(path, str)
            or not path
            or not isinstance(name, str)
            or not name
            or not isinstance(target, dict)
        ):
            raise ValueError("Missing shortcut identity or target")
        kind = target.get("type")
        if (
            not isinstance(kind, str)
            or re.fullmatch(r"[A-Za-z][A-Za-z0-9]{0,63}", kind) is None
        ):
            raise ValueError("Invalid shortcut target type")
        record = {
            "shortcutId": shortcut_fingerprint([path, name]),
            "targetType": kind,
            "targetStatus": "unavailable",
            "targetWorkspaceId": None,
            "targetItemId": None,
            "targetReference": None,
            "isShortcutTransform": value.get("isShortcutTransform"),
        }
        detail_key = "oneLake" if kind == "OneLake" else _SHORTCUT_TYPES.get(kind)
        if detail_key is None:
            record["targetStatus"] = "unsupported"
        elif detail_key in target:
            detail = target[detail_key]
            if not isinstance(detail, dict):
                raise TypeError("Invalid shortcut target details")
            if kind == "OneLake":
                for field in ("workspaceId", "itemId"):
                    if field in detail:
                        str(UUID(detail[field]))
                if "workspaceId" in detail and "itemId" in detail:
                    record.update(
                        targetWorkspaceId=str(UUID(detail["workspaceId"])),
                        targetItemId=str(UUID(detail["itemId"])),
                        targetStatus="configured_item",
                    )
            else:
                location = {}
                for field in ("location", "subpath", "path", "bucket", "connectionId"):
                    if field not in detail:
                        continue
                    content = detail[field]
                    if not isinstance(content, str) or not content:
                        raise ValueError("Invalid shortcut external reference")
                    if field == "connectionId":
                        content = str(UUID(content))
                    elif field == "location":
                        parsed = urlsplit(content)
                        if (
                            parsed.scheme != "https"
                            or not parsed.netloc
                            or parsed.username is not None
                            or parsed.password is not None
                        ):
                            raise ValueError("Invalid shortcut external location")
                        content = parsed._replace(query="", fragment="").geturl()
                    location[field] = content
                if "location" in location:
                    record.update(
                        targetReference=shortcut_fingerprint([kind, location]),
                        targetStatus="external_reference",
                    )
        identity = record["shortcutId"]
        if identity in selected and selected[identity] != record:
            raise ValueError("Conflicting shortcut identity")
        selected[identity] = record
    return validate_shortcut_records([selected[key] for key in sorted(selected)])


def shortcut_progress(saved, excluded):
    settings = {"excludeWorkspaceIds": excluded}
    if saved is not None and saved != {"finished": True}:
        try:
            previous = saved["settings"]["excludeWorkspaceIds"]
            if not isinstance(previous, list) or previous != shortcut_exclusions(
                ",".join(previous)
            ):
                raise ValueError("Invalid shortcut progress settings")
        except (KeyError, ValueError, TypeError, AttributeError):
            raise ValueError("Invalid shortcut progress; requests blocked") from None
    if saved is None or saved == {"finished": True} or saved["settings"] != settings:
        return {
            "settings": settings,
            "phase": "inventory",
            "items": [],
            "nextItem": 0,
            "records": [],
            "continuationToken": None,
            "tokens": [],
            "healthy": True,
        }
    try:
        items = shortcut_inventory(
            [{"id": item["itemId"], **item} for item in saved["items"]], excluded
        )
        token, tokens = saved["continuationToken"], saved["tokens"]
        if (
            items != saved["items"]
            or saved["phase"] not in {"inventory", "shortcuts"}
            or type(saved["nextItem"]) is not int
            or not 0 <= saved["nextItem"] <= len(items)
            or type(saved["healthy"]) is not bool
            or not isinstance(tokens, list)
            or len(tokens) != len(set(tokens))
            or len(tokens) > 10000
            or any(not isinstance(value, str) or not value for value in tokens)
            or (
                token is not None
                and (not isinstance(token, str) or not token or tokens[-1:] != [token])
            )
            or (token is None and tokens)
            or (
                saved["phase"] == "inventory"
                and (saved["nextItem"] != 0 or saved["records"])
            )
        ):
            raise ValueError("Invalid shortcut progress")
        validate_shortcut_records(saved["records"])
    except (KeyError, ValueError, TypeError, AttributeError):
        raise ValueError("Invalid shortcut progress; requests blocked") from None
    return saved


def shortcut_scope(execution):
    excluded = execution["excludeWorkspaceIds"]
    if (
        not isinstance(excluded, list)
        or excluded != shortcut_exclusions(",".join(excluded))
        or type(execution["sweepComplete"]) is not bool
        or type(execution["sweepHealthy"]) is not bool
    ):
        raise ValueError("Invalid shortcut execution scope")
    return "shortcut:" + shortcut_fingerprint(excluded)


EVENTSTREAM_MODULE = "fabriceventstreamtopology"
_EVENTSTREAM_GROUPS = {
    "sources": "source",
    "streams": "stream",
    "operators": "operator",
    "destinations": "destination",
}
_EVENTSTREAM_ITEM_TYPES = {
    "source": frozenset(
        {
            "CustomStreamConnector",
            "LakehouseChangeFeed",
            "MirroredDatabaseChangeFeed",
            "BusinessEvents",
            "FabricAnomalyDetectionEvents",
            "FabricJobEvents",
            "FabricOneLakeEvents",
            "FabricWorkspaceItemEvents",
        }
    ),
    "destination": frozenset(
        {"Lakehouse", "Eventhouse", "Notebook", "BusinessEvents", "Activator"}
    ),
}
_EVENTSTREAM_STATUSES = frozenset(
    {
        "Running",
        "Stopped",
        "Failed",
        "Starting",
        "Stopping",
        "Updating",
        "Created",
        "Disabled",
        "Enabled",
        "External",
        "Unknown",
        "Other",
    }
)
_EVENTSTREAM_NODE_FIELDS = {
    "nodeId",
    "role",
    "type",
    "reportedStatus",
    "inputsKnown",
    "targetWorkspaceId",
    "targetItemId",
    "connectionReference",
    "targetStatus",
}


def eventstream_hash(value):
    return hashlib.sha256(
        json.dumps(
            value, sort_keys=True, separators=(",", ":"), allow_nan=False
        ).encode()
    ).hexdigest()


def eventstream_exclusions(value):
    if not isinstance(value, str):
        raise TypeError("Eventstream workspace exclusions must be a string")
    try:
        return sorted(
            {str(UUID(part.strip())) for part in value.split(",") if part.strip()}
        )
    except (ValueError, AttributeError):
        raise ValueError("Eventstream workspace exclusions require UUIDs") from None


def eventstream_inventory(values, excluded):
    if not isinstance(values, list):
        raise TypeError("Invalid Eventstream inventory")
    selected = {}
    for value in values:
        if not isinstance(value, dict) or value.get("type") != "Eventstream":
            raise ValueError("Invalid Eventstream inventory type")
        parent = {
            "workspaceId": str(UUID(value["workspaceId"])),
            "itemId": str(UUID(value["id"])),
            "type": "Eventstream",
        }
        if parent["workspaceId"] not in excluded:
            selected[parent["workspaceId"], parent["itemId"]] = parent
    return [selected[key] for key in sorted(selected)]


def eventstream_scope(execution):
    excluded = execution["excludeWorkspaceIds"]
    if (
        not isinstance(excluded, list)
        or excluded != eventstream_exclusions(",".join(excluded))
        or type(execution["sweepComplete"]) is not bool
        or type(execution["sweepHealthy"]) is not bool
        or not isinstance(execution["sweepId"], str)
        or re.fullmatch(r"[a-f0-9]{32}", execution["sweepId"]) is None
    ):
        raise ValueError("Invalid Eventstream execution scope")
    return "eventstream:" + eventstream_hash([excluded, execution["sweepId"]])


def validate_eventstream_topology(value):
    if (
        not isinstance(value, dict)
        or set(value) != {"schemaVersion", "nodes", "links", "diagnostics"}
        or type(value["schemaVersion"]) is not int
        or value["schemaVersion"] != 1
        or any(
            not isinstance(value[key], list)
            for key in ("nodes", "links", "diagnostics")
        )
    ):
        raise ValueError("Invalid minimized Eventstream topology")
    nodes = {}
    for node in value["nodes"]:
        if (
            not isinstance(node, dict)
            or set(node) != _EVENTSTREAM_NODE_FIELDS
            or not isinstance(node["nodeId"], str)
            or re.fullmatch(r"[a-f0-9]{64}", node["nodeId"]) is None
            or node["nodeId"] in nodes
            or node["role"] not in _EVENTSTREAM_GROUPS.values()
            or not isinstance(node["type"], str)
            or re.fullmatch(r"[A-Za-z][A-Za-z0-9]{0,95}", node["type"]) is None
            or node["reportedStatus"] is not None
            and node["reportedStatus"] not in _EVENTSTREAM_STATUSES
            or type(node["inputsKnown"]) is not bool
            or node["targetStatus"]
            not in {
                "configured_item",
                "connection_reference",
                "unresolved",
                "unsupported",
                "not_applicable",
            }
        ):
            raise ValueError("Invalid topology node")
        nodes[node["nodeId"]] = node
        configured = node["targetStatus"] == "configured_item"
        for field in ("targetWorkspaceId", "targetItemId"):
            identity = node[field]
            if configured:
                if not isinstance(identity, str) or str(UUID(identity)) != identity:
                    raise ValueError("Invalid configured topology target")
            elif identity is not None:
                raise ValueError("Unresolved topology has an item identity")
        reference = node["connectionReference"]
        if reference is not None and (
            not isinstance(reference, str)
            or re.fullmatch(r"[a-f0-9]{64}", reference) is None
        ):
            raise ValueError("Invalid topology connection fingerprint")
        if (
            configured
            and node["type"] not in _EVENTSTREAM_ITEM_TYPES.get(node["role"], ())
            or node["targetStatus"] == "connection_reference"
            and reference is None
            or node["role"] in {"stream", "operator"}
            and (node["targetStatus"] != "not_applicable" or reference is not None)
            or node["role"] in {"source", "destination"}
            and node["targetStatus"] == "not_applicable"
        ):
            raise ValueError("Topology node and target disagree")
    seen = set()
    for link in value["links"]:
        if (
            not isinstance(link, dict)
            or set(link) != {"fromNode", "toNode", "resolved"}
            or any(
                not isinstance(link[key], str)
                or re.fullmatch(r"[a-f0-9]{64}", link[key]) is None
                for key in ("fromNode", "toNode")
            )
            or type(link["resolved"]) is not bool
            or link["toNode"] not in nodes
            or not nodes[link["toNode"]]["inputsKnown"]
            or link["resolved"] != (link["fromNode"] in nodes)
            or (link["fromNode"], link["toNode"]) in seen
        ):
            raise ValueError("Invalid topology link")
        seen.add((link["fromNode"], link["toNode"]))
    expected = []
    for node in nodes.values():
        if not node["inputsKnown"] and node["role"] != "source":
            expected.append(
                {"nodeId": node["nodeId"], "reason": "missing_topology_inputs"}
            )
        if node["targetStatus"] in {"unresolved", "unsupported"}:
            expected.append(
                {
                    "nodeId": node["nodeId"],
                    "reason": "unresolved_topology_target"
                    if node["targetStatus"] == "unresolved"
                    else "unsupported_topology_target",
                }
            )
    for link in value["links"]:
        if not link["resolved"]:
            expected.append(
                {
                    "nodeId": link["toNode"],
                    "reason": "unresolved_topology_input",
                    "inputNodeId": link["fromNode"],
                }
            )
    if sorted(value["diagnostics"], key=eventstream_hash) != sorted(
        expected, key=eventstream_hash
    ):
        raise ValueError("Topology diagnostics disagree with evidence")
    return value


def eventstream_topology(body):
    """Keep structural nodes/inputs and selected UUIDs; never keep node payloads."""
    if (
        not isinstance(body, dict)
        or body.get("continuationToken") is not None
        or body.get("continuationUri") is not None
    ):
        raise ValueError("Invalid or unexpectedly paginated topology")
    nodes, names, ids, inputs = [], set(), set(), []
    for group, role in _EVENTSTREAM_GROUPS.items():
        values = body.get(group)
        if not isinstance(values, list):
            raise TypeError("Missing or malformed topology node collection")
        for value in values:
            if not isinstance(value, dict):
                raise TypeError("Invalid topology node")
            name, kind = value.get("name"), value.get("type")
            if (
                not isinstance(name, str)
                or not name
                or name in names
                or not isinstance(kind, str)
                or re.fullmatch(r"[A-Za-z][A-Za-z0-9]{0,95}", kind) is None
            ):
                raise ValueError("Invalid or ambiguous topology node identity")
            names.add(name)
            if "id" in value:
                identity = str(UUID(value["id"]))
                if identity in ids:
                    raise ValueError("Duplicate topology node UUID")
                ids.add(identity)
            properties = value.get("properties", {})
            if not isinstance(properties, dict):
                raise TypeError("Invalid topology node properties")
            status = value.get("status")
            if status is not None and not isinstance(status, str):
                raise TypeError("Invalid topology node status")
            node = {
                "nodeId": eventstream_hash(name),
                "role": role,
                "type": kind,
                "reportedStatus": status
                if status in _EVENTSTREAM_STATUSES or status is None
                else "Other",
                "inputsKnown": "inputNodes" in value,
                "targetWorkspaceId": None,
                "targetItemId": None,
                "connectionReference": None,
                "targetStatus": "not_applicable"
                if role in {"stream", "operator"}
                else "unresolved",
            }
            if role in {"source", "destination"}:
                if "dataConnectionId" in properties:
                    node["connectionReference"] = eventstream_hash(
                        str(UUID(properties["dataConnectionId"]))
                    )
                    node["targetStatus"] = "connection_reference"
                if kind in _EVENTSTREAM_ITEM_TYPES[role]:
                    for field in ("workspaceId", "itemId"):
                        if field in properties:
                            str(UUID(properties[field]))
                    if "workspaceId" in properties and "itemId" in properties:
                        node.update(
                            targetWorkspaceId=str(UUID(properties["workspaceId"])),
                            targetItemId=str(UUID(properties["itemId"])),
                            targetStatus="configured_item",
                        )
                elif "workspaceId" in properties or "itemId" in properties:
                    node["targetStatus"] = "unsupported"
            if "inputNodes" in value:
                if not isinstance(value["inputNodes"], list):
                    raise TypeError("Invalid topology inputs")
                for entry in value["inputNodes"]:
                    if (
                        not isinstance(entry, dict)
                        or not isinstance(entry.get("name"), str)
                        or not entry["name"]
                    ):
                        raise ValueError("Invalid topology input")
                    inputs.append((eventstream_hash(entry["name"]), node["nodeId"]))
            nodes.append(node)
    identities = {node["nodeId"] for node in nodes}
    links = [
        {"fromNode": source, "toNode": destination, "resolved": source in identities}
        for source, destination in sorted(set(inputs))
    ]
    diagnostics = []
    for node in nodes:
        if not node["inputsKnown"] and node["role"] != "source":
            diagnostics.append(
                {"nodeId": node["nodeId"], "reason": "missing_topology_inputs"}
            )
        if node["targetStatus"] in {"unresolved", "unsupported"}:
            diagnostics.append(
                {
                    "nodeId": node["nodeId"],
                    "reason": "unresolved_topology_target"
                    if node["targetStatus"] == "unresolved"
                    else "unsupported_topology_target",
                }
            )
    diagnostics.extend(
        {
            "nodeId": link["toNode"],
            "inputNodeId": link["fromNode"],
            "reason": "unresolved_topology_input",
        }
        for link in links
        if not link["resolved"]
    )
    return validate_eventstream_topology(
        {
            "schemaVersion": 1,
            "nodes": sorted(nodes, key=lambda node: node["nodeId"]),
            "links": links,
            "diagnostics": sorted(diagnostics, key=eventstream_hash),
        }
    )


def eventstream_progress(saved, excluded):
    if saved is not None and saved != {"finished": True}:
        try:
            execution = {
                **saved["settings"],
                "sweepComplete": False,
                "sweepHealthy": True,
                "sweepId": saved["sweepId"],
            }
            eventstream_scope(execution)
        except (KeyError, ValueError, TypeError, AttributeError):
            raise ValueError("Invalid Eventstream progress; requests blocked") from None
    settings = {"excludeWorkspaceIds": excluded}
    if saved is None or saved == {"finished": True} or saved["settings"] != settings:
        return {
            "settings": settings,
            "phase": "inventory",
            "items": [],
            "nextItem": 0,
            "continuationToken": None,
            "tokens": [],
            "healthy": True,
            "sweepId": uuid4().hex,
        }
    try:
        items = eventstream_inventory(
            [{"id": item["itemId"], **item} for item in saved["items"]], excluded
        )
        token, tokens = saved["continuationToken"], saved["tokens"]
        if (
            items != saved["items"]
            or saved["phase"] not in {"inventory", "topology"}
            or type(saved["nextItem"]) is not int
            or not 0 <= saved["nextItem"] <= len(items)
            or type(saved["healthy"]) is not bool
            or not isinstance(tokens, list)
            or len(tokens) != len(set(tokens))
            or len(tokens) > 10000
            or any(not isinstance(value, str) or not value for value in tokens)
            or token is not None
            and (not isinstance(token, str) or not token or tokens[-1:] != [token])
            or token is None
            and tokens
            or saved["phase"] == "inventory"
            and saved["nextItem"] != 0
            or saved["phase"] == "topology"
            and (token is not None or tokens)
        ):
            raise ValueError("Invalid Eventstream progress")
    except (KeyError, ValueError, TypeError, AttributeError):
        raise ValueError("Invalid Eventstream progress; requests blocked") from None
    return saved


SCHEMA_VERSION = 1
LAKEHOUSE_FILES = "/lakehouse/default/Files"
PERMISSIONS = {
    "fabricitemaccesspermissions": "Scoped",
    "fabricitemaccesspermissions_tenant": "Tenant",
}
MODULES = (
    *PERMISSIONS,
    "fabricitemjobhistory",
    "fabricitemdefinitions",
    "fabricconnections",
    "fabricdeploymentpipelineoperations",
)
CORE_SOURCE_PREFIXES = {*MODULES, "items", "workspaces", "catalog", "lineage_edges"}
SHORTCUT_PREFIXES = {SHORTCUT_MODULE}
EVENTSTREAM_PREFIXES = {EVENTSTREAM_MODULE}
POWER_BI = {"Report", "Dashboard", "SemanticModel", "App", "Dataflow"}
COMMON = {
    "schema_version": "int",
    "observation_id": "string",
    "source_path": "string",
    "source_sha256": "string",
    "record_sha256": "string",
    "observed_at_utc": "string",
    "module": "string",
    "mode": "string",
    "run_id": "string",
    "cycle_id": "string",
    "coverage_status": "string",
}
TARGET_COLUMNS = {
    "workspace_id": "string",
    "item_id": "string",
    "item_type": "string",
}
SCHEMAS = {
    "collection_health": {
        "complete": "boolean",
        "finished": "boolean",
        "successful_resources": "long",
        "failed_resources": "long",
        "record_count": "long",
        "coverage": "string",
        "issue": "string",
    },
    "inventory": {
        **TARGET_COLUMNS,
        "item_name": "string",
        "inventory_source": "string",
    },
    "permission_cycles": {
        "chain_valid": "boolean",
        "cycle_finished": "boolean",
        "cycle_healthy": "boolean",
        "slice_count": "long",
        "selected_items": "long",
    },
    "permission_resources": {
        **TARGET_COLUMNS,
        "observation_unit": "string",
        "record_count": "long",
        "resource_outcome": "string",
    },
    "permission_grants": {
        **TARGET_COLUMNS,
        "principal_id": "string",
        "principal_type": "string",
        "permissions_json": "string",
        "additional_permissions_json": "string",
    },
    "job_resources": {**TARGET_COLUMNS, "record_count": "long"},
    "job_observations": {
        **TARGET_COLUMNS,
        "job_id": "string",
        "job_type": "string",
        "invoke_type": "string",
        "status": "string",
        "start_time_utc": "string",
        "end_time_utc": "string",
        "duration_seconds": "double",
        "root_activity_id": "string",
        "failure_code": "string",
    },
    "lineage_resources": {
        **TARGET_COLUMNS,
        "evidence_type": "string",
        "record_count": "long",
        "diagnostic_count": "long",
    },
    "lineage_edges": {
        **TARGET_COLUMNS,
        "related_workspace_id": "string",
        "related_item_id": "string",
        "related_type": "string",
        "direction": "string",
        "relation_type": "string",
        "reference": "string",
        "resolution": "string",
        "evidence_type": "string",
        "runtime_verified": "boolean",
    },
    "lineage_diagnostics": {
        **TARGET_COLUMNS,
        "reason": "string",
        "location": "string",
    },
}
CORE_DATASETS = tuple(SCHEMAS)
ESTATE_SCHEMAS = {
    "estate_coverage": {
        "analysis": "string",
        "source_date": "string",
        "record_count": "long",
        "issue": "string",
    },
    "activity_events": {
        **TARGET_COLUMNS,
        "event_id": "string",
        "event_time_utc": "string",
        "operation": "string",
        "workload": "string",
        "actor_key": "string",
        "actor_type": "string",
        "is_success": "boolean",
        "source_date": "string",
    },
    "catalog_workspaces": {
        "workspace_id": "string",
        "workspace_name": "string",
        "workspace_type": "string",
        "state": "string",
        "capacity_id": "string",
        "returned_collections_json": "string",
        "source_date": "string",
    },
    "catalog_items": {
        **TARGET_COLUMNS,
        "item_name": "string",
        "source_date": "string",
    },
    "catalog_models": {
        **TARGET_COLUMNS,
        "model_name": "string",
        "schema_coverage": "string",
        "table_count": "long",
        "relationship_count": "long",
        "source_date": "string",
    },
    "model_tables": {
        **TARGET_COLUMNS,
        "table_name": "string",
        "is_hidden": "boolean",
        "column_count": "long",
        "measure_count": "long",
        "source_date": "string",
    },
    "model_columns": {
        **TARGET_COLUMNS,
        "table_name": "string",
        "column_name": "string",
        "data_type": "string",
        "is_hidden": "boolean",
        "source_date": "string",
    },
    "model_measures": {
        **TARGET_COLUMNS,
        "table_name": "string",
        "measure_name": "string",
        "data_type": "string",
        "expression_returned": "boolean",
        "source_date": "string",
    },
    "model_relationships": {
        **TARGET_COLUMNS,
        "from_table": "string",
        "from_column": "string",
        "to_table": "string",
        "to_column": "string",
        "from_cardinality": "string",
        "to_cardinality": "string",
        "is_active": "boolean",
        "source_date": "string",
    },
    "connection_observations": {
        "connection_id": "string",
        "connection_name": "string",
        "gateway_id": "string",
        "connectivity_type": "string",
        "privacy_level": "string",
        "connection_type": "string",
    },
    "gateway_observations": {
        "gateway_id": "string",
        "gateway_name": "string",
        "gateway_type": "string",
        "source_date": "string",
    },
    "gateway_datasources": {
        "gateway_id": "string",
        "datasource_id": "string",
        "datasource_name": "string",
        "datasource_type": "string",
        "detail_returned": "boolean",
        "source_date": "string",
    },
    "gateway_status": {
        "gateway_id": "string",
        "datasource_id": "string",
        "status_code": "string",
        "http_status": "long",
        "connectivity_observation": "string",
        "source_date": "string",
    },
    "refresh_observations": {
        **TARGET_COLUMNS,
        "dataset_name": "string",
        "refresh_id": "string",
        "refresh_type": "string",
        "status": "string",
        "start_time_utc": "string",
        "end_time_utc": "string",
        "duration_seconds": "double",
        "failure_code": "string",
        "source_date": "string",
    },
    "refreshable_observations": {
        **TARGET_COLUMNS,
        "dataset_name": "string",
        "capacity_id": "string",
        "refresh_count": "long",
        "refresh_failures": "long",
        "schedule_enabled": "boolean",
        "source_date": "string",
    },
}
SCHEMAS.update(ESTATE_SCHEMAS)
ESTATE_DATASETS = tuple(SCHEMAS)
ESTATE_PREFIXES = {"activity", "gateways", "datasetrefresh", "datasetrefreshable"}
MODULE_PREFIXES = {
    "capacity",
    "capacity_metrics",
    "graph",
    "roles",
    "tenant",
    "delegatedsettings",
    "domains",
    "gitconnections",
    "tags",
    "widelysharedlinks",
}
MODULE_SCHEMAS = {
    "apps": {
        "app_id": "string",
        "app_name": "string",
        "workspace_id": "string",
        "last_update_utc": "string",
        "source_date": "string",
    },
    "capacities": {
        "capacity_id": "string",
        "capacity_name": "string",
        "sku": "string",
        "state": "string",
        "region": "string",
        "source_date": "string",
    },
    "capacity_metric_observations": {
        "capacity_id": "string",
        "sku": "string",
        "metric_group": "string",
        "metric_date": "string",
        "timepoint_raw": "string",
        "timepoint_utc": "string",
        "metric_name": "string",
        "metric_value": "double",
        "source_date": "string",
    },
    "directory_users": {
        "principal_id": "string",
        "account_enabled": "boolean",
        "user_type": "string",
        "assigned_license_count": "long",
        "source_date": "string",
    },
    "directory_groups": {
        "group_id": "string",
        "security_enabled": "boolean",
        "mail_enabled": "boolean",
        "group_types_json": "string",
        "is_audience_group": "boolean",
        "source_date": "string",
    },
    "group_memberships": {
        "group_id": "string",
        "principal_id": "string",
        "membership_kind": "string",
        "is_audience_group": "boolean",
        "source_date": "string",
    },
    "user_licenses": {
        "principal_id": "string",
        "sku_id": "string",
        "disabled_plans_json": "string",
        "source_date": "string",
    },
    "subscribed_skus": {
        "sku_id": "string",
        "sku_part_number": "string",
        "capability_status": "string",
        "consumed_units": "long",
        "enabled_units": "long",
        "suspended_units": "long",
        "warning_units": "long",
        "source_date": "string",
    },
    "workspace_role_scopes": {
        "workspace_id": "string",
        "roles_returned": "boolean",
        "assignment_count": "long",
        "source_date": "string",
    },
    "workspace_role_assignments": {
        "workspace_id": "string",
        "principal_id": "string",
        "principal_type": "string",
        "workspace_role": "string",
        "source_date": "string",
    },
    "tenant_settings": {
        "setting_name": "string",
        "title": "string",
        "setting_group": "string",
        "enabled": "boolean",
        "can_specify_groups": "boolean",
        "enabled_group_count": "long",
        "excluded_group_count": "long",
        "source_date": "string",
    },
    "delegated_setting_scopes": {
        "scope_type": "string",
        "entity_id": "string",
        "settings_returned": "boolean",
        "setting_count": "long",
        "source_date": "string",
    },
    "delegated_setting_overrides": {
        "scope_type": "string",
        "entity_id": "string",
        "setting_name": "string",
        "title": "string",
        "setting_group": "string",
        "enabled": "boolean",
        "delegated_from": "string",
        "enabled_group_count": "long",
        "excluded_group_count": "long",
        "source_date": "string",
    },
    "tenant_setting_groups": {
        "scope_type": "string",
        "entity_id": "string",
        "setting_name": "string",
        "group_id": "string",
        "group_rule": "string",
        "source_date": "string",
    },
    "domains": {
        "domain_id": "string",
        "domain_name": "string",
        "parent_domain_id": "string",
        "source_date": "string",
    },
    "domain_workspaces": {
        "domain_id": "string",
        "workspace_id": "string",
        "workspace_name": "string",
        "source_date": "string",
    },
    "workspace_git_connections": {
        "workspace_id": "string",
        "provider_type": "string",
        "owner_key": "string",
        "organization_name": "string",
        "project_name": "string",
        "repository_name": "string",
        "branch_name": "string",
        "directory_key": "string",
        "connection_state": "string",
        "last_sync_utc": "string",
        "source_date": "string",
    },
    "tags": {
        "tag_id": "string",
        "tag_name": "string",
        "scope_type": "string",
        "domain_id": "string",
        "source_date": "string",
    },
    "organization_shares": {
        "artifact_id": "string",
        "artifact_type": "string",
        "access_right": "string",
        "share_type": "string",
        "source_date": "string",
    },
    "publish_to_web_shares": {
        "artifact_id": "string",
        "artifact_type": "string",
        "access_right": "string",
        "share_type": "string",
        "source_date": "string",
    },
    "deployment_pipelines": {
        "pipeline_id": "string",
        "pipeline_name": "string",
        "source_date": "string",
    },
    "deployment_operation_resources": {
        "pipeline_id": "string",
        "record_count": "long",
        "resource_outcome": "string",
        "source_date": "string",
    },
    "deployment_operations": {
        "pipeline_id": "string",
        "operation_id": "string",
        "operation_type": "string",
        "status": "string",
        "last_updated_utc": "string",
        "execution_start_utc": "string",
        "execution_end_utc": "string",
        "duration_seconds": "double",
        "source_stage_id": "string",
        "target_stage_id": "string",
        "source_date": "string",
    },
}
SCHEMAS.update(MODULE_SCHEMAS)
MODULE_DATASETS = tuple(SCHEMAS)
PIPELINE_ACTIVITY_MODULE = "fabricpipelineactivityruns"
PIPELINE_ACTIVITY_PREFIXES = {PIPELINE_ACTIVITY_MODULE}
PIPELINE_ACTIVITY_SCHEMAS = {
    "pipeline_activity_resources": {
        **TARGET_COLUMNS,
        "scope_id": "string",
        "pipeline_run_id": "string",
        "observation_unit": "string",
        "last_updated_after_utc": "string",
        "last_updated_before_utc": "string",
        "record_count": "long",
        "issue": "string",
    },
    "pipeline_activity_observations": {
        **TARGET_COLUMNS,
        "scope_id": "string",
        "pipeline_run_id": "string",
        "activity_run_id": "string",
        "activity_name": "string",
        "activity_type": "string",
        "status": "string",
        "last_updated_after_utc": "string",
        "last_updated_before_utc": "string",
        "start_time_utc": "string",
        "end_time_utc": "string",
        "duration_ms": "double",
        "rows_read": "long",
        "rows_copied": "long",
        "data_read": "long",
        "data_written": "long",
        "throughput": "double",
        "child_pipeline_run_id": "string",
    },
}
SCHEMAS.update(PIPELINE_ACTIVITY_SCHEMAS)


def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)


def sha256(data):
    return hashlib.sha256(data).hexdigest()


def utc(value):
    if not isinstance(value, str):
        raise TypeError("Invalid UTC timestamp")
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        raise ValueError("Naive timestamp")
    return (
        parsed.astimezone(timezone.utc)
        .isoformat(timespec="microseconds")
        .replace("+00:00", "Z")
    )


def text(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError("Missing text")
    return value


def uuid(value):
    if not isinstance(value, str) or str(UUID(value)) != value:
        raise ValueError("Invalid canonical identity")
    return value


def records(value):
    if not isinstance(value, list) or any(not isinstance(v, dict) for v in value):
        raise ValueError("Invalid record collection")
    return value


def target(parent):
    return uuid(parent["workspaceId"]), uuid(parent["itemId"]), text(parent["type"])


def target_fields(parent):
    return dict(zip(TARGET_COLUMNS, target(parent), strict=True))


def document(path, data):
    """Parse bytes without propagating source text into diagnostics."""
    relative = PurePosixPath(path)
    if relative.is_absolute() or ".." in relative.parts or "\\" in path:
        raise ValueError("Unsafe source path")
    try:
        value = json.loads(data)
        canonical(value)
        issue = None
    except (ValueError, UnicodeError, RecursionError):
        value, issue = None, "malformed_json"
    return {
        "path": path,
        "sha256": sha256(data),
        "value": value,
        "issue": issue,
    }


def load_documents(
    raw_root,
    *,
    max_files=50000,
    max_bytes=128 * 1024 * 1024,
    max_file_bytes=8 * 1024 * 1024,
    source_prefixes=None,
    allow_empty=False,
):
    """Bounded driver-side reads of the attached Lakehouse; never write raw files."""
    root = Path(raw_root)
    allowed = Path(LAKEHOUSE_FILES)
    if (
        not root.is_absolute()
        or ".." in root.parts
        or root.resolve() == allowed
        or allowed not in root.resolve().parents
        or not root.is_dir()
    ):
        raise ValueError(
            "Select an existing folder inside the attached Lakehouse Files"
        )
    result, total = [], 0
    prefixes = (
        CORE_SOURCE_PREFIXES
        | ESTATE_PREFIXES
        | MODULE_PREFIXES
        | PIPELINE_ACTIVITY_PREFIXES
        | SHORTCUT_PREFIXES
        | EVENTSTREAM_PREFIXES
    )
    if source_prefixes is not None:
        selected = set(source_prefixes)
        if not selected or not selected.issubset(prefixes):
            raise ValueError("Select known source folders")
        prefixes = selected
    for path in sorted(root.rglob("*.json")):
        relative = path.relative_to(root)
        if relative.parts[0] not in prefixes or path.name in {
            "request_gate.json",
            "progress.json",
            "tenant_progress.json",
        }:
            continue
        if relative.parts[0] == "gateways" and path.name == "users.json":
            continue
        if path.is_symlink() or root.resolve() not in path.resolve().parents:
            raise ValueError("Source path escaped the raw folder")
        size = path.stat().st_size
        total += size
        if len(result) >= max_files or total > max_bytes or size > max_file_bytes:
            raise ValueError("Raw evidence exceeds configured driver safety limits")
        data = path.read_bytes()
        if len(data) != size:
            raise ValueError("Source changed during reading")
        result.append(document(relative.as_posix(), data))
    if not result and not allow_empty:
        raise ValueError("No supported collection files; refusing an empty import")
    return result


def row(table, doc, key, *, observed=None, **values):
    path = doc["path"]
    parts = PurePosixPath(path).parts
    value = doc["value"] if isinstance(doc["value"], dict) else {}
    timestamp = observed if observed is not None else value.get("observedAt")
    try:
        timestamp = utc(timestamp)
    except (ValueError, TypeError):
        timestamp = None
    result = dict.fromkeys({**COMMON, **SCHEMAS[table]})
    result.update(
        schema_version=SCHEMA_VERSION,
        observation_id=sha256(canonical([table, path, key]).encode()),
        source_path=path,
        source_sha256=doc["sha256"],
        observed_at_utc=timestamp,
        module=parts[0],
        mode=PERMISSIONS.get(parts[0]),
        run_id=next(
            (
                value[key]
                for key in ("runId", "scanId")
                if isinstance(value.get(key), str)
            ),
            None,
        ),
        coverage_status="success",
    )
    if set(values) - set(result):
        raise ValueError("Unknown contract column")
    result.update(values)
    if parts[0] in {"items", "workspaces", "catalog"} or result["coverage_status"] in {
        "invalid",
        "unfinished",
        "legacy_unverified",
    }:
        result["observation_id"] = sha256(
            canonical([table, path, key, doc["sha256"]]).encode()
        )
    return result


def _validate_records(module, resource):
    parent = resource["parent"]
    inventory = parent.get("observationUnit") == "inventoryPage"
    if module in PERMISSIONS:
        if inventory:
            if not isinstance(parent.get("inventoryComplete"), bool):
                raise ValueError("Invalid inventory completion")
            if PERMISSIONS[module] == "Scoped":
                uuid(parent["workspaceId"])
        else:
            target(parent)
    elif module == "fabricitemjobhistory":
        if target(parent)[2] not in {"Notebook", "DataPipeline", "SparkJobDefinition"}:
            raise ValueError("Unsupported job item")
    elif (
        module == "fabricitemdefinitions"
        and parent
        and target(parent)[2] not in {"Notebook", "DataPipeline"}
    ):
        raise ValueError("Unsupported definition item")
    elif module == SHORTCUT_MODULE and parent and target(parent)[2] != "Lakehouse":
        raise ValueError("Unsupported shortcut parent")
    elif module == EVENTSTREAM_MODULE and parent and target(parent)[2] != "Eventstream":
        raise ValueError("Unsupported Eventstream parent")
    if not resource["complete"]:
        text(resource["error"])
        if "records" in resource or (inventory and parent["inventoryComplete"]):
            raise ValueError("Failed resource contains records or complete inventory")
        return
    if "error" in resource:
        raise ValueError("Successful resource contains an error")
    values = records(resource["records"])
    if module == SHORTCUT_MODULE:
        if parent:
            validate_shortcut_records(values)
        elif (
            shortcut_inventory(
                [{"id": value["itemId"], **value} for value in values], []
            )
            != values
        ):
            raise ValueError("Invalid shortcut inventory identities")
    elif module == EVENTSTREAM_MODULE:
        if parent:
            if len(values) != 1:
                raise ValueError("Missing or repeated Eventstream topology")
            validate_eventstream_topology(values[0])
        elif (
            eventstream_inventory(
                [{"id": value["itemId"], **value} for value in values], []
            )
            != values
        ):
            raise ValueError("Invalid Eventstream inventory identities")
    identities = []
    for value in values:
        if module in PERMISSIONS and inventory:
            identity = target(value)
            if PERMISSIONS[module] == "Scoped" and identity[0] != parent["workspaceId"]:
                raise ValueError("Inventory workspace mismatch")
            identities.append(
                identity[:2] if PERMISSIONS[module] == "Tenant" else identity
            )
        elif module in PERMISSIONS:
            principal, detail = value["principal"], value["itemAccessDetails"]
            text(principal["id"])
            if not isinstance(principal["type"], str):
                raise ValueError("Invalid principal type")
            if not (
                detail["type"] == parent["type"]
                or (parent["type"] in POWER_BI and detail["type"] is None)
            ):
                raise ValueError("Permission item type mismatch")
            for field in ("permissions", "additionalPermissions"):
                if field == "additionalPermissions" and (
                    field not in detail
                    or (detail[field] is None and parent["type"] in POWER_BI)
                ):
                    continue
                if not isinstance(detail[field], list) or any(
                    not isinstance(v, str) for v in detail[field]
                ):
                    raise ValueError("Invalid permission list")
        elif module == "fabricitemjobhistory":
            for field in ("id", "jobType", "invokeType", "status"):
                text(value[field])
            if value["itemId"] != parent["itemId"]:
                raise ValueError("Job parent mismatch")
            for field in ("startTimeUtc", "endTimeUtc"):
                if value.get(field) is not None:
                    utc(value[field])
            if value.get("failureReason") is not None and not isinstance(
                value["failureReason"], dict
            ):
                raise ValueError("Invalid job failure metadata")
            if value.get("rootActivityId") is not None and not isinstance(
                value["rootActivityId"], str
            ):
                raise ValueError("Invalid activity identity")
            identities.append(value["id"])
    if len(identities) != len(set(identities)):
        raise ValueError("Duplicate observation identity")


def _resource_name_endpoint(module, resource):
    parent = resource["parent"]
    if module in PERMISSIONS:
        if parent.get("observationUnit") == "inventoryPage":
            if PERMISSIONS[module] == "Tenant":
                return "inventory.json", "admin/items?state=active"
            workspace = parent["workspaceId"]
            return f"inventory_{workspace}.json", f"admin/items?workspaceId={workspace}"
        workspace, item, kind = target(parent)
        suffix = f"?type={kind}" if kind in POWER_BI else ""
        if kind == "Dataflow" and resource.get("requestPath") == (
            f"admin/workspaces/{workspace}/items/{item}/users"
        ):
            suffix = ""
        return (
            f"{workspace}_{item}.json",
            f"admin/workspaces/{workspace}/items/{item}/users{suffix}",
        )
    if module == "fabricitemjobhistory":
        workspace, item, _ = target(parent)
        return (
            f"{workspace}_{item}.json",
            f"workspaces/{workspace}/items/{item}/jobs/instances",
        )
    if module == "fabricitemdefinitions":
        if not parent:
            return "inventory.json", "admin/items"
        workspace, item, kind = target(parent)
        endpoint = (
            f"notebooks/{item}/getDefinition?format=ipynb"
            if kind == "Notebook"
            else f"dataPipelines/{item}/getDefinition"
        )
        return f"{workspace}_{item}.json", f"workspaces/{workspace}/{endpoint}"
    if module == SHORTCUT_MODULE:
        if not parent:
            return "inventory.json", "admin/items?type=Lakehouse"
        workspace, item, _ = target(parent)
        return (
            f"{workspace}_{item}.json",
            f"workspaces/{workspace}/items/{item}/shortcuts",
        )
    if module == EVENTSTREAM_MODULE:
        if not parent:
            return "inventory.json", "admin/items?type=Eventstream"
        workspace, item, _ = target(parent)
        return (
            f"{workspace}_{item}.json",
            f"workspaces/{workspace}/eventstreams/{item}/topology",
        )
    return None, None


def build_runs(
    documents, *, modules=MODULES, resource_validator=None, manifest_validator=None
):
    grouped = {}
    for doc in documents:
        path = PurePosixPath(doc["path"])
        if path.parts[0] in modules:
            grouped.setdefault(str(path.parent), {})[path.name] = doc
    result = {}
    for folder, files in sorted(grouped.items()):
        module = folder.split("/")[0]
        manifest_doc = files.get("manifest.json")
        representative = manifest_doc or files[min(files)]
        run = {
            "doc": representative,
            "module": module,
            "manifest": None,
            "resources": [],
            "valid": False,
            "issue": None,
            "chain": None,
        }
        result[folder + "/manifest.json"] = run
        try:
            parts = folder.split("/")
            if len(parts) != 5 or not re.fullmatch(r"[a-f0-9]{32}", parts[-1]):
                raise ValueError("Invalid immutable run path")
            datetime.strptime("/".join(parts[1:4]), "%Y/%m/%d")
            if manifest_doc is None or manifest_doc["issue"]:
                raise ValueError("Missing or malformed manifest")
            manifest = manifest_doc["value"]
            if not isinstance(manifest, dict):
                raise TypeError("Invalid manifest object")
            run["manifest"] = manifest
            if (
                manifest["runId"] != parts[-1]
                or not isinstance(manifest["complete"], bool)
                or not isinstance(manifest["finished"], bool)
                or not isinstance(manifest["errors"], list)
            ):
                raise ValueError("Invalid manifest fields")
            final_time = utc(manifest["observedAt"])
            entries = records(manifest["resources"])
            names = []
            if not manifest["finished"]:
                if manifest["complete"] or entries or manifest["errors"]:
                    raise ValueError("Invalid initial manifest")
                run["valid"] = True
                continue
            if manifest_validator is not None:
                manifest_validator(manifest)
            if module == SHORTCUT_MODULE:
                shortcut_scope(manifest["execution"])
            elif module == EVENTSTREAM_MODULE:
                eventstream_scope(manifest["execution"])
            for entry in entries:
                name = text(entry["file"])
                if (
                    PurePosixPath(name).name != name
                    or "\\" in name
                    or name == "manifest.json"
                    or not name.endswith(".json")
                    or name in names
                    or not isinstance(entry["complete"], bool)
                    or not isinstance(entry["parent"], dict)
                ):
                    raise ValueError("Invalid manifest entry")
                names.append(name)
                doc = files[name]
                resource = doc["value"]
                if (
                    doc["issue"]
                    or not isinstance(resource, dict)
                    or resource["runId"] != manifest["runId"]
                    or not isinstance(resource["complete"], bool)
                    or resource["parent"] != entry["parent"]
                    or resource["complete"] != entry["complete"]
                    or utc(resource["observedAt"]) > final_time
                ):
                    raise ValueError("Resource and manifest disagree")
                if resource_validator is not None:
                    expected_name, endpoint = resource_validator(resource, manifest)
                else:
                    _validate_records(module, resource)
                    expected_name, endpoint = _resource_name_endpoint(module, resource)
                if module in {SHORTCUT_MODULE, EVENTSTREAM_MODULE}:
                    excluded = manifest["execution"]["excludeWorkspaceIds"]
                    parents = (
                        [resource["parent"]]
                        if resource["parent"]
                        else resource.get("records", [])
                    )
                    if any(parent["workspaceId"] in excluded for parent in parents):
                        raise ValueError("Excluded configuration workspace captured")
                if endpoint is not None and (
                    name != expected_name or resource["requestPath"] != endpoint
                ):
                    raise ValueError("Resource identity or endpoint mismatch")
                run["resources"].append(doc)
            if set(files) != {"manifest.json", *names}:
                raise ValueError("Unlisted or missing run resources")
            if module in PERMISSIONS:
                if len(entries) != 1:
                    raise ValueError("Permission slice must contain one resource")
            elif module == "fabricitemjobhistory" and (
                not 1 <= len(entries) <= 100
                or len({target(entry["parent"]) for entry in entries}) != len(entries)
            ):
                raise ValueError("Invalid job resource scope")
            elif module in {SHORTCUT_MODULE, EVENTSTREAM_MODULE}:
                execution = manifest["execution"]
                if manifest["complete"] != (
                    execution["sweepComplete"]
                    and execution["sweepHealthy"]
                    and all(entry["complete"] for entry in entries)
                    and not manifest["errors"]
                ):
                    raise ValueError(
                        "Configuration sweep outcome disagrees with resources"
                    )
            elif module == PIPELINE_ACTIVITY_MODULE:
                execution = manifest.get("execution") or {}
                if execution.get("mode") not in {"discovery", "explicit"}:
                    raise ValueError("Invalid pipeline activity collection mode")
                if execution["mode"] == "discovery":
                    if (
                        type(execution.get("sweepComplete")) is not bool
                        or type(execution.get("sweepHealthy")) is not bool
                    ):
                        raise ValueError("Invalid pipeline discovery outcome")
                    expected_complete = (
                        execution["sweepComplete"] and execution["sweepHealthy"]
                    )
                    if manifest["complete"] != (
                        expected_complete
                        and all(entry["complete"] for entry in entries)
                        and not manifest["errors"]
                    ):
                        raise ValueError(
                            "Pipeline sweep outcome disagrees with resources"
                        )
                elif manifest["complete"] != (
                    all(entry["complete"] for entry in entries)
                    and not manifest["errors"]
                ):
                    raise ValueError(
                        "Pipeline explicit outcome disagrees with resources"
                    )
            elif manifest["complete"] != (
                all(entry["complete"] for entry in entries) and not manifest["errors"]
            ):
                raise ValueError("Manifest completeness disagrees with resources")
            run["valid"] = True
        except (ValueError, KeyError, TypeError, AttributeError, IndexError):
            run["issue"] = "invalid_run_artifacts"
    for path, run in result.items():
        if (
            run["module"] in PERMISSIONS
            and run["valid"]
            and run["manifest"]["finished"]
        ):
            try:
                run["chain"] = permission_chain(path, result)
            except (ValueError, KeyError, TypeError, AttributeError, IndexError):
                run["issue"] = "invalid_permission_chain"
                run["valid"] = False
    return result


def _integer(value):
    if type(value) is not int or value < 0:
        raise ValueError("Invalid nonnegative integer")
    return value


def _permission_fallback(manifest, resource, prior):
    e = manifest["execution"]
    mode = e.get("dataflowFallback")
    if mode is not None and mode not in ("pending", "request"):
        raise ValueError("Invalid Dataflow fallback mode")
    previous = prior["execution"] if prior is not None else {}
    expected = previous.get("dataflowFallback") == "pending" or (
        previous.get("dataflowFallback") == "request"
        and previous["deferred"]
        and previous["attempts"] == 0
    )
    if (mode == "request") != expected:
        raise ValueError("Unlinked Dataflow fallback")
    parent = resource["parent"]
    if mode is None:
        if parent.get("type") == "Dataflow" and not resource["requestPath"].endswith(
            "?type=Dataflow"
        ):
            raise ValueError("Unmarked untyped Dataflow request")
        return False
    workspace, item, kind = target(parent)
    endpoint = f"admin/workspaces/{workspace}/items/{item}/users"
    if kind != "Dataflow" or resource["requestPath"] != (
        endpoint + ("?type=Dataflow" if mode == "pending" else "")
    ):
        raise ValueError("Invalid Dataflow fallback resource")
    pending = mode == "pending"
    if pending and (
        resource["complete"]
        or e["attempts"] != 1
        or not e["deferred"]
        or manifest["errors"]
        not in [
            [
                {
                    "error": "Fabric request failed",
                    "status": 404,
                    "errorCode": code,
                }
            ]
            for code in ("EntityNotFound", "ItemNotFound")
        ]
    ):
        raise ValueError("Invalid Dataflow fallback admission")
    return pending


def permission_chain(path, runs):
    """Follow enumerated immutable manifests only, then validate cursor transitions."""
    module = runs[path]["module"]
    tenant = PERMISSIONS[module] == "Tenant"
    chain, seen = [], set()
    while True:
        if path in seen or len(seen) >= 10000 or path not in runs:
            raise ValueError("Missing, cyclic or excessive permission evidence")
        seen.add(path)
        run = runs[path]
        manifest = run["manifest"]
        if not run["valid"] or run["module"] != module or not manifest["finished"]:
            raise ValueError("Uncommitted permission evidence")
        chain.append(run)
        execution = manifest["execution"]
        links = execution["cycleEvidence"]
        if not isinstance(links, list) or len(links) > 1:
            raise ValueError("Invalid evidence links")
        if not links:
            break
        path = text(links[0])
    chain.reverse()
    selected, selected_keys, inventories, access = [], set(), set(), []
    before, healthy, prior, prior_parent, prior_task = 0, True, None, None, None
    task_keys, task_ids = set(), set()
    inventory_workspaces = set()
    for run in chain:
        manifest, resource = run["manifest"], run["resources"][0]["value"]
        e, parent = manifest["execution"], resource["parent"]
        inventory = parent.get("observationUnit") == "inventoryPage"
        for field in ("cycleFinished", "cycleHealthy", "deferred"):
            if not isinstance(e[field], bool):
                raise TypeError("Invalid cycle flag")
        attempts = _integer(e["attempts"])
        if attempts > 1:
            raise ValueError("Invalid slice attempt count")
        fallback_pending = _permission_fallback(manifest, resource, prior)
        unattempted = e["deferred"] and attempts == 0
        if resource["complete"] and (attempts != 1 or e["deferred"]):
            raise ValueError("Successful deferred or unattempted resource")
        if prior is not None and (
            prior["execution"]["cycleFinished"]
            or utc(prior["observedAt"]) > utc(resource["observedAt"])
        ):
            raise ValueError("Invalid cycle transition")
        previous_healthy = healthy
        healthy = healthy and (resource["complete"] or unattempted or fallback_pending)
        if e["cycleHealthy"] != healthy or (
            healthy and manifest["errors"] and not unattempted and not fallback_pending
        ):
            raise ValueError("Nonmonotonic cycle health")
        if tenant:
            if (
                e["scopeMode"] != "tenant"
                or e["phase"] != ("inventory" if inventory else "access")
                or not isinstance(e["inventoryComplete"], bool)
                or e["resourceOutcome"]
                not in {
                    "succeeded",
                    "failed",
                    "unsupported",
                    "denied",
                    "not_found",
                    "deferred",
                }
                or (e["resourceOutcome"] == "succeeded") != resource["complete"]
                or (e["resourceOutcome"] == "deferred") != e["deferred"]
                or (attempts == 0 and not e["deferred"])
            ):
                raise ValueError("Invalid tenant execution")
            if inventory:
                if before != 0 or (
                    prior is not None and prior["execution"]["inventoryComplete"]
                ):
                    raise ValueError("Inventory after tenant access")
                if resource["complete"]:
                    for item in resource["records"]:
                        key = target(item)[:2]
                        if key in selected_keys:
                            raise ValueError("Duplicate tenant inventory")
                        selected_keys.add(key)
                        selected.append(item)
                after = 0
                if e["inventoryComplete"] != parent["inventoryComplete"]:
                    raise ValueError("Inventory completion mismatch")
            else:
                if (
                    prior is None
                    or not prior["execution"]["inventoryComplete"]
                    or before >= len(selected)
                    or target(parent) != target(selected[before])
                    or not e["inventoryComplete"]
                ):
                    raise ValueError("Access outside tenant inventory order")
                after = before + (0 if unattempted or fallback_pending else 1)
            if (
                _integer(e["discoveredItems"]) != len(selected)
                or _integer(e["nextItem"]) != after
                or e["cycleFinished"]
                != (e["inventoryComplete"] and after == len(selected))
            ):
                raise ValueError("Invalid tenant cursor")
        else:
            count = _integer(e["taskCount"])
            if not 2 <= count <= 100 or (
                prior is not None and count != prior["execution"]["taskCount"]
            ):
                raise ValueError("Invalid scoped task count")
            workspace = parent["workspaceId"]
            key = (workspace, "inventory") if inventory else target(parent)
            if prior is None:
                if not inventory:
                    raise ValueError("Cycle does not begin with inventory")
            elif before == prior_task:
                prior_key = (
                    (prior_parent["workspaceId"], "inventory")
                    if prior_parent.get("observationUnit") == "inventoryPage"
                    else target(prior_parent)
                )
                if key != prior_key:
                    raise ValueError("Deferred task changed identity")
            elif prior_parent.get("observationUnit") == "inventoryPage":
                if inventory or workspace != prior_parent["workspaceId"]:
                    raise ValueError("Invalid inventory to access transition")
            elif not inventory:
                if workspace != prior_parent["workspaceId"]:
                    raise ValueError("Access workspace changed without inventory")
            elif workspace in inventory_workspaces:
                raise ValueError("Repeated workspace inventory task")
            if inventory:
                inventory_workspaces.add(workspace)
                if resource["complete"]:
                    for item in resource["records"]:
                        identity = target(item)
                        if identity in selected_keys:
                            raise ValueError("Duplicate scoped inventory")
                        selected_keys.add(identity)
                        selected.append(item)
                    if parent["inventoryComplete"]:
                        inventories.add(workspace)
            elif resource["complete"]:
                if target(parent) not in selected_keys or workspace not in inventories:
                    raise ValueError("Access outside completed scoped inventory")
                access.append(target(parent))
            delta = (
                0
                if unattempted
                or fallback_pending
                or (
                    inventory
                    and resource["complete"]
                    and not parent["inventoryComplete"]
                )
                else 1
            )
            after = before + delta
            if (
                _integer(e["nextTask"]) != after
                or after > count
                or e["cycleFinished"] != (after == count)
            ):
                raise ValueError("Invalid scoped cursor")
            task_keys.add((before, key))
            task_ids.add(before)
            if manifest["complete"] and (
                not 1 <= len(access) <= 50
                or len(access) != len(set(access))
                or set(access) != selected_keys
                or len(task_keys) != count
                or len(task_ids) != count
                or len({key[0] for key in access}) + len(access) != count
            ):
                raise ValueError("Incomplete scoped selection")
        if manifest["complete"] != (e["cycleFinished"] and healthy):
            raise ValueError("Invalid permission cycle completeness")
        # A failed slice cannot become healthy merely by following a newer link.
        if not previous_healthy and healthy:
            raise ValueError("Cycle health recovered without a new cycle")
        prior_task, before, prior, prior_parent = before, after, manifest, parent
    return {
        "nodes": chain,
        "cycle_id": chain[0]["manifest"]["runId"],
        "selected_items": len(selected),
    }


def _health(run):
    manifest = run["manifest"] or {}
    valid, finished = run["valid"], manifest.get("finished") is True
    resources = run["resources"]
    status = (
        "invalid"
        if not valid
        else "unfinished"
        if not finished
        else "success"
        if manifest["complete"]
        else "partial_or_failed"
    )
    return row(
        "collection_health",
        run["doc"],
        "run",
        coverage_status=status,
        complete=manifest.get("complete") if valid else None,
        finished=manifest.get("finished") if valid else None,
        successful_resources=sum(d["value"]["complete"] for d in resources)
        if valid and finished
        else None,
        failed_resources=sum(not d["value"]["complete"] for d in resources)
        if valid and finished
        else None,
        coverage=manifest.get("coverage")
        if isinstance(manifest.get("coverage"), str)
        else None,
        issue=run["issue"],
        cycle_id=run["chain"]["cycle_id"] if run["chain"] else None,
        mode=shortcut_scope(manifest["execution"])
        if run["module"] == SHORTCUT_MODULE and valid and finished
        else eventstream_scope(manifest["execution"])
        if run["module"] == EVENTSTREAM_MODULE and valid and finished
        else PERMISSIONS.get(run["module"]),
    )


def _permission_tables(runs, output):
    for run in runs.values():
        if run["module"] not in PERMISSIONS:
            continue
        manifest = run["manifest"] or {}
        chain = run["chain"]
        valid = run["valid"] and chain is not None
        cycle_id = (
            chain["cycle_id"]
            if valid
            else (
                manifest.get("runId")
                if isinstance(manifest.get("runId"), str)
                else None
            )
        )
        execution = manifest.get("execution") or {}
        output["permission_cycles"].append(
            row(
                "permission_cycles",
                run["doc"],
                "cycle_tip",
                cycle_id=cycle_id,
                chain_valid=valid,
                cycle_finished=execution.get("cycleFinished") if valid else None,
                cycle_healthy=execution.get("cycleHealthy") if valid else None,
                slice_count=len(chain["nodes"]) if valid else None,
                selected_items=chain["selected_items"] if valid else None,
                coverage_status="success"
                if valid and manifest["complete"]
                else "partial_or_failed"
                if valid
                else "invalid",
            )
        )
        if not valid:
            continue
        doc = run["resources"][0]
        resource, parent = doc["value"], doc["value"]["parent"]
        inventory = parent.get("observationUnit") == "inventoryPage"
        fields = (
            {"workspace_id": parent.get("workspaceId")}
            if inventory
            else target_fields(parent)
        )
        successful = resource["complete"]
        output["permission_resources"].append(
            row(
                "permission_resources",
                doc,
                "resource",
                **fields,
                cycle_id=cycle_id,
                observation_unit="inventory_page" if inventory else "item_access",
                record_count=len(resource["records"]) if successful else None,
                resource_outcome=execution.get("resourceOutcome")
                or (
                    "succeeded"
                    if successful
                    else "deferred"
                    if execution["deferred"]
                    else "failed"
                ),
                coverage_status="success"
                if successful
                else "deferred"
                if execution["deferred"]
                else "failed",
            )
        )
        if not successful:
            continue
        for record in resource["records"]:
            if inventory:
                output["inventory"].append(
                    row(
                        "inventory",
                        doc,
                        target(record),
                        **target_fields(record),
                        inventory_source="permission_inventory",
                        cycle_id=cycle_id,
                    )
                )
            else:
                principal, detail = record["principal"], record["itemAccessDetails"]
                permissions = sorted(set(detail["permissions"]))
                additional = detail.get("additionalPermissions")
                additional = sorted(set(additional)) if additional is not None else None
                output["permission_grants"].append(
                    row(
                        "permission_grants",
                        doc,
                        [principal["id"], principal["type"], permissions, additional],
                        **fields,
                        cycle_id=cycle_id,
                        principal_id=principal["id"],
                        principal_type=principal["type"],
                        permissions_json=canonical(permissions),
                        additional_permissions_json=canonical(additional),
                    )
                )


def _job_tables(runs, output):
    for run in runs.values():
        if run["module"] != "fabricitemjobhistory" or not run["valid"]:
            continue
        if not run["manifest"]["finished"]:
            continue
        for doc in run["resources"]:
            resource, parent = doc["value"], doc["value"]["parent"]
            fields = target_fields(parent)
            output["job_resources"].append(
                row(
                    "job_resources",
                    doc,
                    "resource",
                    **fields,
                    coverage_status="success" if resource["complete"] else "failed",
                    record_count=len(resource["records"])
                    if resource["complete"]
                    else None,
                )
            )
            if not resource["complete"]:
                continue
            for job in resource["records"]:
                start = utc(job["startTimeUtc"]) if job.get("startTimeUtc") else None
                end = utc(job["endTimeUtc"]) if job.get("endTimeUtc") else None
                duration = (
                    (
                        datetime.fromisoformat(end.replace("Z", "+00:00"))
                        - datetime.fromisoformat(start.replace("Z", "+00:00"))
                    ).total_seconds()
                    if start and end
                    else None
                )
                failure = job.get("failureReason") or {}
                output["job_observations"].append(
                    row(
                        "job_observations",
                        doc,
                        job["id"],
                        **fields,
                        job_id=job["id"],
                        job_type=job["jobType"],
                        invoke_type=job["invokeType"],
                        status=job["status"],
                        start_time_utc=start,
                        end_time_utc=end,
                        duration_seconds=duration
                        if duration is not None and duration >= 0
                        else None,
                        root_activity_id=job.get("rootActivityId"),
                        failure_code=failure.get("errorCode")
                        if isinstance(failure.get("errorCode"), str)
                        else None,
                    )
                )


def _legacy_inventory(doc, output):
    path = doc["path"]
    module = path.split("/")[0]
    try:
        if not re.fullmatch(r"(items|workspaces)/\d{4}/\d{2}/\d{2}/[^/]+\.json", path):
            raise ValueError("Invalid legacy inventory path")
        values = records(doc["value"])
        datetime.strptime("/".join(path.split("/")[1:4]), "%Y/%m/%d")
        pending = []
        for value in values:
            item = uuid(value["id"])
            workspace = item if module == "workspaces" else uuid(value["workspaceId"])
            kind = "Workspace" if module == "workspaces" else text(value["type"])
            name = value.get("displayName", value.get("name"))
            pending.append(
                row(
                    "inventory",
                    doc,
                    [workspace, item, kind],
                    workspace_id=workspace,
                    item_id=item,
                    item_type=kind,
                    item_name=name if isinstance(name, str) else None,
                    inventory_source="legacy_" + module,
                    coverage_status="legacy_unverified",
                )
            )
        output["inventory"].extend(pending)
        status, count = "legacy_unverified", len(values)
    except (ValueError, KeyError, TypeError, AttributeError):
        status, count = "invalid", None
    output["collection_health"].append(
        row(
            "collection_health",
            doc,
            "legacy_file",
            coverage_status=status,
            record_count=count,
            issue="no_manifest_or_exact_observation_time",
        )
    )


def _scanner_tables(doc, output, *, legacy_snapshot=None):
    try:
        legacy = legacy_snapshot is not None
        scan = legacy_snapshot if legacy else doc["value"]
        if (
            doc["issue"]
            or (
                not legacy
                and not re.fullmatch(
                    r"lineage_edges/\d{4}/\d{2}/\d{2}/[A-Za-z0-9][A-Za-z0-9_-]{0,127}\.lineage_edges\.json",
                    doc["path"],
                )
            )
            or type(scan["schemaVersion"]) is not int
            or scan["schemaVersion"] != 1
            or scan["source"] != "Scanner"
            or scan["complete"] is not True
            or (
                not legacy
                and PurePosixPath(doc["path"]).name
                != scan["scanId"] + ".lineage_edges.json"
            )
            or scan["scanMode"] not in {"full", "incremental"}
        ):
            raise ValueError("Invalid scanner envelope")
        observed = None if legacy else utc(scan["observedAtUtc"])
        status = "legacy_unverified" if legacy else "success"
        date_parts = (
            doc["path"].split("/")[2:5] if legacy else doc["path"].split("/")[1:4]
        )
        datetime.strptime("/".join(date_parts), "%Y/%m/%d")
        requested = {uuid(value) for value in scan["requestedWorkspaceIds"]}
        returned = {uuid(value) for value in scan["returnedWorkspaceIds"]}
        if not returned.issubset(requested):
            raise ValueError("Unexpected returned workspace")
        items = records(scan["items"])
        indexed = {}
        inventory = []
        for item in items:
            identity, workspace = uuid(item["id"]), uuid(item["workspaceId"])
            if identity in indexed or workspace not in returned:
                raise ValueError("Invalid scanner item inventory")
            indexed[identity] = item
            inventory.append(
                row(
                    "inventory",
                    doc,
                    identity,
                    observed=observed,
                    workspace_id=workspace,
                    item_id=identity,
                    item_type=text(item["type"]),
                    item_name=item.get("name")
                    if isinstance(item.get("name"), str)
                    else None,
                    inventory_source="legacy_scanner" if legacy else "scanner_batch",
                    coverage_status=status,
                )
            )
        edges = []
        for edge in records(scan["edges"]):
            item = indexed[edge["QueryItemId"]]
            if (
                edge["Source"] != "Scanner"
                or edge["Direction"] not in {"upstream", "downstream"}
                or edge["QueryWorkspaceId"] != item["workspaceId"]
                or edge["QueryItemType"] != item["type"]
            ):
                raise ValueError("Invalid scanner edge scope")
            related = text(edge["RelatedItemId"])
            if edge.get("RelatedItemType") is not None:
                text(edge["RelatedItemType"])
            if edge.get("RelatedWorkspaceId") is not None:
                uuid(edge["RelatedWorkspaceId"])
            key = [item["id"], edge["Direction"], related, text(edge["RelationType"])]
            edges.append(
                row(
                    "lineage_edges",
                    doc,
                    key,
                    observed=observed,
                    workspace_id=item["workspaceId"],
                    item_id=item["id"],
                    item_type=item["type"],
                    related_workspace_id=edge.get("RelatedWorkspaceId"),
                    related_item_id=related,
                    related_type=edge.get("RelatedItemType"),
                    direction=edge["Direction"],
                    relation_type=edge["RelationType"],
                    evidence_type="scanner_relationship",
                    runtime_verified=False,
                    coverage_status=status,
                    resolution="in_batch"
                    if related in indexed
                    else "external_or_unresolved",
                )
            )
        coverage = status if requested == returned else "partial_batch"
        resources = [
            row(
                "lineage_resources",
                doc,
                ["scanner_item", identity],
                observed=observed,
                workspace_id=item["workspaceId"],
                item_id=identity,
                item_type=item["type"],
                evidence_type="scanner_relationship",
                record_count=sum(edge["item_id"] == identity for edge in edges),
                diagnostic_count=0,
                coverage_status=coverage,
            )
            for identity, item in indexed.items()
        ]
        output["inventory"].extend(inventory)
        output["lineage_edges"].extend(edges)
        output["lineage_resources"].extend(resources)
        output["collection_health"].append(
            row(
                "collection_health",
                doc,
                "scanner_batch",
                observed=observed,
                complete=None if legacy else requested == returned,
                finished=None if legacy else True,
                record_count=len(items),
                coverage_status=coverage,
                coverage="Scanner batch only; not exhaustive tenant or runtime I/O coverage",
                issue="no_manifest_or_exact_observation_time" if legacy else None,
            )
        )
    except (ValueError, KeyError, TypeError, AttributeError):
        output["collection_health"].append(
            row(
                "collection_health",
                doc,
                "scanner_batch",
                coverage_status="invalid",
                issue="invalid_scanner_envelope",
            )
        )


def _notebook_configuration_tables(runs, output):
    for run in runs.values():
        if (
            run["module"] != "fabricitemdefinitions"
            or not run["valid"]
            or not run["manifest"]["finished"]
        ):
            continue
        for doc in run["resources"]:
            resource, parent = doc["value"], doc["value"]["parent"]
            if parent.get("type") != "Notebook":
                continue
            fields = target_fields(parent)
            status, relationships, diagnostics = "failed", [], []
            if resource["complete"]:
                try:
                    values = records(resource["records"])
                    if len(values) != 1:
                        raise ValueError("Invalid definition record count")
                    decoded = values[0].get("decodedParts", {})
                    if not isinstance(decoded, dict):
                        raise TypeError("Invalid decoded definition")
                    if "notebook-content.ipynb" not in decoded:
                        status = "missing"
                        diagnostics = [
                            {
                                "fieldPath": "/decodedParts/notebook-content.ipynb",
                                "reason": "missing_notebook_configuration",
                            }
                        ]
                    else:
                        analysis = notebook_lakehouse_relationships(
                            decoded["notebook-content.ipynb"]
                        )
                        diagnostics = analysis["diagnostics"]
                        status = "success" if analysis["complete"] else "invalid"
                        if analysis["complete"]:
                            relationships = analysis["relationships"]
                except (ValueError, KeyError, TypeError, AttributeError):
                    status = "invalid"
                    diagnostics = [
                        {
                            "fieldPath": "/decodedParts",
                            "reason": "invalid_notebook_configuration",
                        }
                    ]
            for relationship in relationships:
                output["lineage_edges"].append(
                    row(
                        "lineage_edges",
                        doc,
                        relationship,
                        **fields,
                        direction="association",
                        related_workspace_id=relationship["workspaceId"],
                        related_item_id=relationship["itemId"],
                        related_type="Lakehouse",
                        relation_type=relationship["relation"],
                        reference=relationship["itemId"],
                        resolution="configured_id",
                        evidence_type="configuration_metadata",
                        runtime_verified=False,
                    )
                )
            for index, diagnostic in enumerate(diagnostics):
                output["lineage_diagnostics"].append(
                    row(
                        "lineage_diagnostics",
                        doc,
                        ["notebook_configuration", index],
                        **fields,
                        reason=diagnostic["reason"],
                        location=canonical({"fieldPath": diagnostic["fieldPath"]}),
                    )
                )
            output["lineage_resources"].append(
                row(
                    "lineage_resources",
                    doc,
                    "notebook_configuration",
                    **fields,
                    coverage_status=status,
                    evidence_type="configuration_metadata",
                    record_count=len(relationships) if status == "success" else None,
                    diagnostic_count=len(diagnostics),
                )
            )


def _definition_tables(runs, output):
    for run in runs.values():
        if (
            run["module"] != "fabricitemdefinitions"
            or not run["valid"]
            or not run["manifest"]["finished"]
        ):
            continue
        for doc in run["resources"]:
            resource, parent = doc["value"], doc["value"]["parent"]
            if not parent:
                continue
            fields = target_fields(parent)
            status, edges, diagnostics = "failed", [], []
            if resource["complete"]:
                try:
                    values = records(resource["records"])
                    if len(values) != 1:
                        raise ValueError("Invalid definition record count")
                    key = (
                        "notebookReferences"
                        if parent["type"] == "Notebook"
                        else "pipelineReferences"
                    )
                    refs = values[0][key]
                    if (
                        type(refs["schemaVersion"]) is not int
                        or refs["schemaVersion"] != 1
                        or refs["evidence"] != "static_inference"
                        or refs["runtimeVerified"] is not False
                    ):
                        raise ValueError("Invalid static evidence")
                    for ref in records(refs["references"]):
                        if ref["operation"] not in {"read", "write", "execute"}:
                            raise ValueError("Invalid static operation")
                        reference = text(ref["reference"])
                        kind = text(ref["kind"])
                        if ref.get("workspaceId") is not None:
                            uuid(ref["workspaceId"])
                        edges.append(
                            row(
                                "lineage_edges",
                                doc,
                                ref,
                                **fields,
                                direction="downstream"
                                if ref["operation"] == "write"
                                else "upstream",
                                related_workspace_id=ref.get("workspaceId"),
                                related_item_id=reference
                                if ref.get("referenceType") == "id"
                                else None,
                                related_type=kind,
                                relation_type=ref["operation"],
                                reference=reference,
                                resolution="unresolved",
                                evidence_type="static_inference",
                                runtime_verified=False,
                            )
                        )
                    for index, diagnostic in enumerate(records(refs["diagnostics"])):
                        diagnostics.append(
                            row(
                                "lineage_diagnostics",
                                doc,
                                index,
                                **fields,
                                reason=text(diagnostic["reason"]),
                                location=canonical(
                                    {
                                        key: diagnostic[key]
                                        for key in ("cell", "line", "fieldPath")
                                        if key in diagnostic
                                    }
                                ),
                            )
                        )
                    status = "success"
                except (ValueError, KeyError, TypeError, AttributeError):
                    status, edges, diagnostics = "invalid", [], []
            output["lineage_resources"].append(
                row(
                    "lineage_resources",
                    doc,
                    "definition",
                    **fields,
                    coverage_status=status,
                    evidence_type="static_inference",
                    record_count=len(edges) if status == "success" else None,
                    diagnostic_count=len(diagnostics) if status == "success" else None,
                )
            )
            output["lineage_edges"].extend(edges)
            output["lineage_diagnostics"].extend(diagnostics)


def _shortcut_tables(runs, output):
    for run in runs.values():
        if (
            run["module"] != SHORTCUT_MODULE
            or not run["valid"]
            or not run["manifest"]["finished"]
        ):
            continue
        mode = shortcut_scope(run["manifest"]["execution"])
        for doc in run["resources"]:
            resource, parent = doc["value"], doc["value"]["parent"]
            fields = target_fields(parent) if parent else {}
            values = (
                resource.get("records", []) if parent and resource["complete"] else []
            )
            diagnostics = 0
            for value in values:
                output["lineage_edges"].append(
                    row(
                        "lineage_edges",
                        doc,
                        value["shortcutId"],
                        **fields,
                        mode=mode,
                        direction="association",
                        related_workspace_id=value["targetWorkspaceId"],
                        related_item_id=value["targetItemId"],
                        related_type="FabricItem"
                        if value["targetType"] == "OneLake"
                        else value["targetType"],
                        relation_type="shortcut_transform_target"
                        if value["isShortcutTransform"]
                        else "shortcut_target",
                        reference=value["targetReference"]
                        or (
                            value["targetItemId"]
                            or "shortcut:"
                            + shortcut_fingerprint(
                                [
                                    parent["workspaceId"],
                                    parent["itemId"],
                                    value["shortcutId"],
                                ]
                            )
                        ),
                        resolution=value["targetStatus"],
                        evidence_type="shortcut_configuration",
                        runtime_verified=False,
                    )
                )
                if value["targetStatus"] in {"unavailable", "unsupported"}:
                    diagnostics += 1
                    output["lineage_diagnostics"].append(
                        row(
                            "lineage_diagnostics",
                            doc,
                            value["shortcutId"],
                            **fields,
                            mode=mode,
                            reason="shortcut_target_" + value["targetStatus"],
                            location=canonical({"shortcutId": value["shortcutId"]}),
                        )
                    )
            output["lineage_resources"].append(
                row(
                    "lineage_resources",
                    doc,
                    "shortcuts" if parent else "inventory",
                    **fields,
                    mode=mode,
                    evidence_type="shortcut_configuration",
                    coverage_status="success" if resource["complete"] else "failed",
                    record_count=len(values)
                    if parent and resource["complete"]
                    else None,
                    diagnostic_count=diagnostics
                    if parent and resource["complete"]
                    else None,
                )
            )


def _eventstream_tables(runs, output):
    for run in runs.values():
        if (
            run["module"] != EVENTSTREAM_MODULE
            or not run["valid"]
            or not run["manifest"]["finished"]
        ):
            continue
        mode = eventstream_scope(run["manifest"]["execution"])
        for doc in run["resources"]:
            resource, parent = doc["value"], doc["value"]["parent"]
            fields = target_fields(parent) if parent else {}
            graph = resource["records"][0] if parent and resource["complete"] else None
            edge_count = 0
            if graph is not None:
                nodes = {node["nodeId"]: node for node in graph["nodes"]}

                def edge(
                    key,
                    relation,
                    reference,
                    *,
                    doc=doc,
                    fields=fields,
                    mode=mode,
                    **related,
                ):
                    output["lineage_edges"].append(
                        row(
                            "lineage_edges",
                            doc,
                            key,
                            **fields,
                            mode=mode,
                            direction="configured_link"
                            if relation == "topology_input"
                            else "association",
                            relation_type=relation,
                            reference=canonical(reference),
                            evidence_type="eventstream_configuration",
                            runtime_verified=False,
                            **related,
                        )
                    )

                for node in nodes.values():
                    reference = {
                        key: node[key]
                        for key in (
                            "nodeId",
                            "role",
                            "type",
                            "reportedStatus",
                            "inputsKnown",
                        )
                    }
                    edge(
                        ["node", node["nodeId"]],
                        "topology_node",
                        reference,
                        related_type="EventstreamNode",
                        resolution="configured_node",
                    )
                    edge_count += 1
                    if node["targetStatus"] in {
                        "configured_item",
                        "connection_reference",
                    }:
                        edge(
                            ["target", node["nodeId"]],
                            "topology_target",
                            {
                                **reference,
                                "connectionReference": node["connectionReference"],
                            },
                            related_type="FabricItem"
                            if node["targetStatus"] == "configured_item"
                            else "ConnectionReference",
                            related_workspace_id=node["targetWorkspaceId"],
                            related_item_id=node["targetItemId"],
                            resolution=node["targetStatus"],
                        )
                        edge_count += 1
                for link in graph["links"]:
                    edge(
                        ["input", link["fromNode"], link["toNode"]],
                        "topology_input",
                        {
                            **link,
                            "fromType": nodes[link["fromNode"]]["type"]
                            if link["resolved"]
                            else "Unknown",
                            "toType": nodes[link["toNode"]]["type"],
                        },
                        related_type="EventstreamNode",
                        resolution="configured_link"
                        if link["resolved"]
                        else "unresolved_input",
                    )
                    edge_count += 1
                for diagnostic in graph["diagnostics"]:
                    output["lineage_diagnostics"].append(
                        row(
                            "lineage_diagnostics",
                            doc,
                            eventstream_hash(diagnostic),
                            **fields,
                            mode=mode,
                            reason=diagnostic["reason"],
                            location=canonical(
                                {
                                    key: value
                                    for key, value in diagnostic.items()
                                    if key != "reason"
                                }
                            ),
                        )
                    )
            output["lineage_resources"].append(
                row(
                    "lineage_resources",
                    doc,
                    "topology" if parent else "inventory",
                    **fields,
                    mode=mode,
                    evidence_type="eventstream_configuration",
                    coverage_status="success" if resource["complete"] else "failed",
                    record_count=edge_count if graph is not None else None,
                    diagnostic_count=len(graph["diagnostics"])
                    if graph is not None
                    else None,
                )
            )


def transform(
    documents,
    *,
    scanner_extractor=None,
    estate_extractor=None,
    module_extractor=None,
    pipeline_extractor=None,
    include_shortcuts=False,
    include_eventstream_topology=False,
):
    """Return historical observations and explicit coverage, never inferred deletions."""
    documents = list(documents)
    if len({doc["path"] for doc in documents}) != len(documents):
        raise ValueError("Duplicate source path")
    output = {table: [] for table in SCHEMAS}
    if (
        type(include_shortcuts) is not bool
        or type(include_eventstream_topology) is not bool
    ):
        raise TypeError("Choose boolean configuration inclusion flags")
    modules = MODULES
    if include_shortcuts:
        modules += (SHORTCUT_MODULE,)
    if include_eventstream_topology:
        modules += (EVENTSTREAM_MODULE,)
    runs = build_runs(documents, modules=modules)
    output["collection_health"].extend(_health(run) for run in runs.values())
    _permission_tables(runs, output)
    _job_tables(runs, output)
    _definition_tables(runs, output)
    _notebook_configuration_tables(runs, output)
    if include_shortcuts:
        _shortcut_tables(runs, output)
    if include_eventstream_topology:
        _eventstream_tables(runs, output)
    present = {doc["path"].split("/")[0] for doc in documents}
    for doc in documents:
        module = doc["path"].split("/")[0]
        if module in {"items", "workspaces"}:
            _legacy_inventory(doc, output)
        elif module == "lineage_edges":
            _scanner_tables(doc, output)
        elif module == "catalog" and doc["path"].startswith("catalog/scans/"):
            try:
                if scanner_extractor is None or not re.fullmatch(
                    r"catalog/scans/\d{4}/\d{2}/\d{2}/\d{8}_\d+\.scanResults(?:\.fullscan)?\.json",
                    doc["path"],
                ):
                    raise ValueError("Unsupported raw scanner artifact")
                scan = doc["value"]
                workspaces = [
                    workspace["id"] for workspace in records(scan["workspaces"])
                ]
                snapshot = scanner_extractor(
                    scan, "legacy_" + doc["sha256"][:32], workspaces, None
                )
                _scanner_tables(doc, output, legacy_snapshot=snapshot)
            except (ValueError, KeyError, TypeError, AttributeError):
                output["collection_health"].append(
                    row(
                        "collection_health",
                        doc,
                        "legacy_scanner",
                        coverage_status="invalid",
                        issue="invalid_raw_scanner",
                    )
                )
    expected_modules = (*MODULES, "lineage_edges", "items", "workspaces")
    if include_shortcuts:
        expected_modules += (SHORTCUT_MODULE,)
    if include_eventstream_topology:
        expected_modules += (EVENTSTREAM_MODULE,)
    for module in expected_modules:
        if module not in present:
            output["collection_health"].append(
                row(
                    "collection_health",
                    {"path": module + "/_missing", "sha256": "", "value": None},
                    "missing",
                    coverage_status="missing",
                    issue="no_collection_files",
                )
            )
    if estate_extractor is not None:
        estate_extractor(documents, runs, output)
    if module_extractor is not None:
        module_extractor(documents, runs, output)
    if pipeline_extractor is not None:
        pipeline_extractor(documents, output)
    for table, rows in output.items():
        unique = {}
        for entry in rows:
            entry["record_sha256"] = sha256(
                canonical(
                    {
                        key: value
                        for key, value in entry.items()
                        if key != "record_sha256"
                    }
                ).encode()
            )
            existing = unique.get(entry["observation_id"])
            if existing is not None and existing != entry:
                raise ValueError("Conflicting observation identity")
            unique[entry["observation_id"]] = entry
        output[table] = sorted(
            unique.values(), key=lambda entry: entry["observation_id"]
        )
    return output


def latest(rows, keys):
    """Select latest observations inside an explicitly chosen scope."""
    selected = {}
    for entry in rows:
        key = tuple(entry[field] for field in keys)
        order = (entry["observed_at_utc"] or "", entry["source_path"])
        if key not in selected or order > (
            selected[key]["observed_at_utc"] or "",
            selected[key]["source_path"],
        ):
            selected[key] = entry
    return list(selected.values())


def selected_permission_cycle(tables, mode):
    if mode not in {"Scoped", "Tenant"}:
        raise ValueError("Select Scoped or Tenant explicitly")
    candidates = [
        entry for entry in tables["permission_cycles"] if entry["mode"] == mode
    ]
    if not candidates:
        return None
    unknown = [entry for entry in candidates if entry["observed_at_utc"] is None]
    if unknown:
        return max(unknown, key=lambda entry: entry["source_path"])
    return max(
        candidates,
        key=lambda entry: (entry["observed_at_utc"] or "", entry["source_path"]),
    )


def table_name(prefix, dataset):
    if not re.fullmatch(r"monitor_[A-Za-z0-9_]*", prefix) or dataset not in SCHEMAS:
        raise ValueError("Use a monitor_ table prefix and a known dataset")
    return prefix + dataset


def to_frame(spark_session, dataset, rows):
    from pyspark.sql import types as T

    types = {
        "int": T.IntegerType,
        "long": T.LongType,
        "double": T.DoubleType,
        "boolean": T.BooleanType,
        "string": T.StringType,
    }
    schema = T.StructType(
        [
            T.StructField(name, types[kind](), True)
            for name, kind in {**COMMON, **SCHEMAS[dataset]}.items()
        ]
    )
    return spark_session.createDataFrame(rows, schema=schema)


def publish(spark_session, tables, datasets, prefix, *, write=False):
    """Insert immutable observations only; never overwrite, delete or run VACUUM."""
    from delta.tables import DeltaTable
    from pyspark.sql import functions as F

    spark_session.conf.set("spark.sql.session.timeZone", "UTC")
    prepared = {}
    for dataset in datasets:
        name = table_name(prefix, dataset)
        frame = to_frame(spark_session, dataset, tables[dataset])
        exists = spark_session.catalog.tableExists(name)
        if exists:
            properties = {
                entry["key"]: entry["value"]
                for entry in spark_session.sql(f"SHOW TBLPROPERTIES {name}").collect()
            }
            if (
                properties.get("monitor.contract") != str(SCHEMA_VERSION)
                or properties.get("monitor.dataset") != dataset
            ):
                raise ValueError("Existing table is not owned by this Monitor contract")
            existing = DeltaTable.forName(spark_session, name).toDF()
            expected_types = {
                field.name: field.dataType for field in frame.schema.fields
            }
            if {
                field.name: field.dataType for field in existing.schema.fields
            } != expected_types:
                raise ValueError("Existing table schema differs; select a new prefix")
            conflict = (
                existing.alias("old")
                .join(frame.alias("new"), "observation_id")
                .filter(
                    ~F.col("old.record_sha256").eqNullSafe(F.col("new.record_sha256"))
                )
            )
            if conflict.limit(1).count():
                raise ValueError(
                    "Immutable observation changed; no overwrite permitted"
                )
            changed_source = (
                existing.alias("old")
                .join(frame.alias("new"), "source_path")
                .filter(
                    ~F.col("old.source_sha256").eqNullSafe(F.col("new.source_sha256"))
                    & ~F.col("old.coverage_status").isin(
                        "invalid", "unfinished", "missing", "legacy_unverified"
                    )
                )
            )
            if changed_source.limit(1).count():
                raise ValueError("Committed source changed; no new version permitted")
        prepared[dataset] = (name, frame, exists)
    if not write:
        print(
            "PREVIEW ONLY: no Delta tables written. Set WRITE_TABLES=True to persist."
        )
        return {dataset: len(tables[dataset]) for dataset in datasets}
    for dataset, (name, frame, exists) in prepared.items():
        if not exists:
            columns = ", ".join(
                f"{column} {kind}"
                for column, kind in {**COMMON, **SCHEMAS[dataset]}.items()
            )
            spark_session.sql(
                f"CREATE TABLE {name} ({columns}) USING DELTA "
                f"TBLPROPERTIES ('monitor.contract'='{SCHEMA_VERSION}', "
                f"'monitor.dataset'='{dataset}')"
            )
        if tables[dataset]:
            DeltaTable.forName(spark_session, name).alias("old").merge(
                frame.alias("new"), "old.observation_id = new.observation_id"
            ).whenNotMatchedInsertAll().execute()
    print("Published immutable observations; no cross-table transaction is promised.")
    return {dataset: len(tables[dataset]) for dataset in datasets}


def read_curated(spark_session, prefix, dataset):
    name = table_name(prefix, dataset)
    if not spark_session.catalog.tableExists(name):
        raise ValueError(
            f"Missing {name}; run its transformation with WRITE_TABLES=True"
        )
    properties = {
        entry["key"]: entry["value"]
        for entry in spark_session.sql(f"SHOW TBLPROPERTIES {name}").collect()
    }
    if (
        properties.get("monitor.contract") != str(SCHEMA_VERSION)
        or properties.get("monitor.dataset") != dataset
    ):
        raise ValueError("Selected table does not match the Monitor contract")
    return spark_session.table(name)


def latest_frame(frame, keys):
    from pyspark.sql import Window
    from pyspark.sql import functions as F

    order = Window.partitionBy(*keys).orderBy(
        F.col("observed_at_utc").desc_nulls_last(), F.col("source_path").desc()
    )
    return (
        frame.withColumn("_rank", F.row_number().over(order))
        .filter(F.col("_rank") == 1)
        .drop("_rank")
    )


def plot_counts(frame, category, title, *, limit=20):
    """Collect only a bounded aggregate, never arbitrary raw principal/job rows."""
    import matplotlib.pyplot as plt
    from pyspark.sql import functions as F

    values = (
        frame.groupBy(category)
        .count()
        .orderBy(F.desc("count"), F.asc(category))
        .limit(limit)
        .toPandas()
    )
    if values.empty:
        print(title + ": no matching observations; inspect the coverage tables.")
        return
    values[category] = values[category].fillna("unknown")
    axis = values.plot.bar(x=category, y="count", legend=False, title=title)
    axis.set_ylabel("Observations")
    plt.tight_layout()
    plt.show()


def preview_inventory(spark_session, display_frame, tables):
    display_frame(
        to_frame(spark_session, "collection_health", tables["collection_health"])
        .groupBy("module", "coverage_status")
        .count()
        .orderBy("module", "coverage_status")
    )
    display_frame(
        to_frame(spark_session, "inventory", tables["inventory"])
        .groupBy("inventory_source", "item_type", "coverage_status")
        .count()
    )


def preview_permissions(spark_session, display_frame, tables, mode):
    selected = selected_permission_cycle(tables, mode)
    if selected is None:
        print("No selected-mode cycle. Missing evidence is not a zero-grant success.")
        return
    print(
        {
            key: selected[key]
            for key in (
                "mode",
                "cycle_id",
                "coverage_status",
                "chain_valid",
                "cycle_finished",
                "cycle_healthy",
                "selected_items",
            )
        }
    )
    resources = [
        entry
        for entry in tables["permission_resources"]
        if entry["mode"] == mode and entry["cycle_id"] == selected["cycle_id"]
    ]
    display_frame(
        to_frame(spark_session, "permission_resources", resources)
        .groupBy("observation_unit", "coverage_status")
        .count()
    )


def preview_jobs(spark_session, display_frame, tables):
    current_jobs = latest(
        tables["job_observations"], ["workspace_id", "item_id", "job_id"]
    )
    display_frame(
        to_frame(spark_session, "job_resources", tables["job_resources"])
        .groupBy("coverage_status")
        .count()
    )
    display_frame(
        to_frame(spark_session, "job_observations", current_jobs)
        .groupBy("item_type", "status")
        .count()
    )


def preview_lineage(spark_session, display_frame, tables):
    for dataset, keys in (
        ("lineage_resources", ["evidence_type", "coverage_status"]),
        ("lineage_edges", ["evidence_type", "direction", "related_type"]),
        ("lineage_diagnostics", ["reason"]),
    ):
        display_frame(
            to_frame(spark_session, dataset, tables[dataset]).groupBy(*keys).count()
        )


def graph_lineage(frame, max_edges):
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    if type(max_edges) is not int or not 1 <= max_edges <= 200:
        raise ValueError("Select 1-200 graph edges")
    values = frame.orderBy("observation_id").limit(max_edges + 1).collect()
    if not values:
        print("No selected lineage edges. Check resource coverage and diagnostics.")
        return
    truncated = len(values) > max_edges
    values = values[:max_edges]
    nodes, edges = {}, []
    for entry in values:
        query = "item:" + entry["item_id"]
        nodes[query] = (entry["item_type"] or "Item") + ":" + entry["item_id"][:8]
        if entry["evidence_type"] == "eventstream_configuration":
            reference = json.loads(entry["reference"])
            scope = "topology:" + entry["workspace_id"] + ":" + entry["item_id"] + ":"
            relation = entry["relation_type"]
            if relation == "topology_input":
                source, destination = (
                    scope + reference["fromNode"],
                    scope + reference["toNode"],
                )
                nodes[source] = reference["fromType"] + ":" + reference["fromNode"][:8]
                nodes[destination] = reference["toType"] + ":" + reference["toNode"][:8]
            else:
                node = scope + reference["nodeId"]
                nodes[node] = reference["type"] + ":" + reference["nodeId"][:8]
                if relation == "topology_node":
                    source, destination = query, node
                else:
                    target = (
                        entry["related_item_id"] or reference["connectionReference"]
                    )
                    destination = (
                        "item:" + target
                        if entry["related_item_id"]
                        else "connection:" + target
                    )
                    nodes.setdefault(
                        destination,
                        (entry["related_type"] or "Reference") + ":" + target[:8],
                    )
                    source = node
            edges.append((source, destination, relation))
            continue
        reference = entry["related_item_id"] or entry["reference"] or "unresolved"
        related = (
            "item:" + entry["related_item_id"]
            if entry["related_item_id"]
            else "reference:" + (entry["related_type"] or "") + ":" + reference
        )
        nodes.setdefault(
            related,
            (entry["related_type"] or "Unresolved")
            + ":"
            + sha256(reference.encode())[:8],
        )
        source, destination = (
            (related, query) if entry["direction"] == "upstream" else (query, related)
        )
        edges.append((source, destination, entry["evidence_type"]))
    ordered = sorted(nodes)
    positions = {
        node: (
            math.cos(2 * math.pi * index / len(ordered)),
            math.sin(2 * math.pi * index / len(ordered)),
        )
        for index, node in enumerate(ordered)
    }
    _, axis = plt.subplots(figsize=(13, 10))
    for source, destination, evidence in edges:
        configured = evidence in {
            "configuration_metadata",
            "shortcut_configuration",
            "topology_node",
            "topology_input",
            "topology_target",
        }
        axis.annotate(
            "",
            xy=positions[destination],
            xytext=positions[source],
            arrowprops={
                "arrowstyle": "->"
                if evidence == "topology_input" or not configured
                else "-",
                "alpha": 0.6,
                "color": "tab:purple"
                if evidence == "topology_input"
                else "tab:gray"
                if evidence == "topology_node"
                else "tab:green"
                if configured
                else "tab:orange"
                if evidence == "static_inference"
                else "tab:blue",
                "linestyle": "--" if configured else "-",
            },
        )
    for node, (x, y) in positions.items():
        axis.scatter([x], [y], s=80)
        axis.text(x * 1.1, y * 1.1, nodes[node], ha="center", fontsize=8)
    axis.legend(
        handles=[
            Line2D([0], [0], color="tab:blue", label="Scanner relationship"),
            Line2D([0], [0], color="tab:orange", label="Potential static reference"),
            Line2D(
                [0],
                [0],
                color="tab:green",
                linestyle="--",
                label="Configured Lakehouse/shortcut association (not I/O)",
            ),
            Line2D(
                [0],
                [0],
                color="tab:purple",
                linestyle="--",
                label="Declared Eventstream input (not runtime)",
            ),
            Line2D(
                [0],
                [0],
                color="tab:gray",
                linestyle="--",
                label="Eventstream node membership",
            ),
        ]
    )
    axis.set_title(
        "Selected relationships (not runtime I/O)"
        + ("; truncated" if truncated else "")
    )
    axis.set_xlim(-1.5, 1.5)
    axis.set_ylim(-1.5, 1.5)
    axis.axis("off")
    plt.show()


def render(
    spark_session,
    display_frame,
    *,
    prefix,
    permission_mode,
    permission_cycle_id="",
    stale_after_hours=24,
    lineage_source_path="",
    max_graph_edges=40,
):
    from pyspark.sql import functions as F

    if permission_mode not in {"Scoped", "Tenant"}:
        raise ValueError("Choose Scoped or Tenant explicitly")
    if isinstance(stale_after_hours, bool) or stale_after_hours <= 0:
        raise ValueError("Choose a positive staleness threshold")
    spark_session.conf.set("spark.sql.session.timeZone", "UTC")
    health = read_curated(spark_session, prefix, "collection_health")
    latest_health = latest_frame(health, ["module"]).withColumn(
        "freshness",
        F.when(F.col("observed_at_utc").isNull(), "unknown")
        .when(
            F.unix_timestamp(F.current_timestamp())
            - F.unix_timestamp(F.to_timestamp("observed_at_utc"))
            > stale_after_hours * 3600,
            "stale",
        )
        .otherwise("within_threshold"),
    )
    display_frame(
        latest_health.select(
            "module", "coverage_status", "freshness", "observed_at_utc", "issue"
        )
    )
    plot_counts(
        health, "coverage_status", "Retained collection observations by coverage"
    )
    print(
        "Historical invalid/unknown observations remain visible, even after later success."
    )
    display_frame(
        health.filter(F.col("coverage_status").isin("invalid", "unfinished", "missing"))
        .groupBy("module", "coverage_status")
        .count()
    )

    inventory = read_curated(spark_session, prefix, "inventory")
    plot_counts(
        inventory, "item_type", "Inventory observations (not current unique items)"
    )

    cycles = read_curated(spark_session, prefix, "permission_cycles").filter(
        F.col("mode") == permission_mode
    )
    if permission_cycle_id:
        cycles = cycles.filter(F.col("cycle_id") == permission_cycle_id)
    unknown_cycle = (
        not permission_cycle_id
        and cycles.filter(F.col("observed_at_utc").isNull()).limit(1).count()
    )
    selected = (
        []
        if unknown_cycle
        else (
            cycles.orderBy(
                F.col("observed_at_utc").desc_nulls_last(), F.col("source_path").desc()
            )
            .limit(1)
            .collect()
        )
    )
    if not selected:
        print(
            "No safely selected permission cycle (missing/unknown evidence). "
            "Inspect health or select an explicit historical cycle; this is not zero grants."
        )
    else:
        selected = selected[0]
        print(
            {
                key: selected[key]
                for key in (
                    "mode",
                    "cycle_id",
                    "coverage_status",
                    "chain_valid",
                    "cycle_finished",
                    "cycle_healthy",
                )
            }
        )
        resources = read_curated(spark_session, prefix, "permission_resources").filter(
            (F.col("mode") == permission_mode)
            & (F.col("cycle_id") == selected["cycle_id"])
            & (F.col("observation_unit") == "item_access")
        )
        resources = latest_frame(resources, ["workspace_id", "item_id"])
        plot_counts(resources, "coverage_status", "Selected-cycle item access coverage")
        display_frame(
            resources.groupBy("coverage_status").agg(
                F.count("*").alias("items"),
                F.sum("record_count").alias("successful_access_records"),
            )
        )
        if not selected["chain_valid"]:
            print("Invalid selected cycle: no grant view is admitted.")
        else:
            grant_keys = ["mode", "cycle_id", "workspace_id", "item_id", "source_path"]
            grants = read_curated(spark_session, prefix, "permission_grants")
            grants = grants.join(
                resources.filter(F.col("coverage_status") == "success")
                .select(*grant_keys)
                .distinct(),
                grant_keys,
                "inner",
            )
            permissions = grants.select(
                F.explode(F.from_json("permissions_json", "array<string>")).alias(
                    "permission"
                )
            )
            plot_counts(
                permissions,
                "permission",
                "Selected-cycle observed permission assignments",
            )

    jobs = read_curated(spark_session, prefix, "job_observations")
    jobs = latest_frame(jobs, ["workspace_id", "item_id", "job_id"])
    plot_counts(
        jobs, "status", "Latest retained status per job (not current polling health)"
    )
    display_frame(
        jobs.groupBy("item_type", "status").agg(
            F.count("*").alias("jobs"),
            F.avg("duration_seconds").alias("mean_known_duration_seconds"),
        )
    )
    job_resources = read_curated(spark_session, prefix, "job_resources")
    job_resources = latest_frame(job_resources, ["workspace_id", "item_id"])
    plot_counts(
        job_resources, "coverage_status", "Latest per-item job collection coverage"
    )

    resources = read_curated(spark_session, prefix, "lineage_resources")
    edges = read_curated(spark_session, prefix, "lineage_edges")
    if lineage_source_path:
        resources = resources.filter(F.col("source_path") == lineage_source_path)
        edges = edges.filter(F.col("source_path") == lineage_source_path)
        print("Explicit historical source selected; this is not a current graph.")
        eligible = resources.filter(
            F.col("coverage_status").isin(
                "success", "legacy_unverified", "partial_batch"
            )
        )
    else:
        resources = latest_frame(
            resources.filter(F.col("observed_at_utc").isNotNull()),
            ["workspace_id", "item_id", "evidence_type"],
        )
        # Select the latest attempt before filtering success; never fall back to old edges.
        eligible = resources.filter(
            F.col("coverage_status").isin("success", "partial_batch")
        )
        for configuration_module in ("fabricshortcuts", "fabriceventstreamtopology"):
            attempt = (
                latest_health.filter(F.col("module") == configuration_module)
                .select("mode", "coverage_status")
                .limit(1)
                .collect()
            )
            if attempt:
                selected = attempt[0]
                eligible = eligible.filter(
                    (F.col("module") != configuration_module)
                    | (
                        (F.col("mode") == selected["mode"])
                        & F.lit(selected["coverage_status"] == "success")
                    )
                )
            else:
                eligible = eligible.filter(F.col("module") != configuration_module)
        uncertain = health.filter(
            F.col("module").isin(
                "fabricitemdefinitions",
                "lineage_edges",
                "fabricshortcuts",
                "fabriceventstreamtopology",
            )
            & F.col("coverage_status").isin("invalid", "unfinished")
        )
        if uncertain.limit(1).count():
            print(
                "Invalid/unfinished lineage evidence prevents automatic graph selection. "
                "Inspect health, then select an explicit historical LINEAGE_SOURCE_PATH."
            )
            eligible = eligible.limit(0)
    plot_counts(resources, "coverage_status", "Selected lineage resource coverage")
    keys = ["source_path", "workspace_id", "item_id", "evidence_type"]
    edges = edges.join(eligible.select(*keys).distinct(), keys, "inner")
    plot_counts(
        edges, "evidence_type", "Selected lineage evidence, never runtime proof"
    )
    diagnostics = read_curated(spark_session, prefix, "lineage_diagnostics")
    plot_counts(diagnostics, "reason", "Retained static-analysis diagnostics")
    graph_lineage(edges, max_graph_edges)

In [ ]:
render(
    spark,
    display,
    prefix=TABLE_PREFIX,
    permission_mode=PERMISSION_MODE,
    permission_cycle_id=PERMISSION_CYCLE_ID,
    stale_after_hours=STALE_AFTER_HOURS,
    lineage_source_path=LINEAGE_SOURCE_PATH,
    max_graph_edges=MAX_GRAPH_EDGES,
)